# LIA / Laya: обучение для muz-bot-ds в Google Colab и Kaggle

**Цель:** дообучить `convaiinnovations/laya-multilingual` на вручную размеченных
русских командах, проверить ошибки и сохранить checkpoint, который загружается через `LAYA_MODEL_PATH`.
Это классификатор вариантов ответа, а не генеративная модель. Whisper и TTS здесь не обучаются.
Результатов обучения пока нет: все выходы появятся после запуска на ваших данных.

Ноутбук самодостаточен: копии функций обработки текста, вопросов, Python-обработчика и
обучающих утилит встроены ниже. Клонирование репозитория и токен Hugging Face не нужны.
Контракт зафиксирован по исходникам; их SHA-256 сохраняются в отчёт.

**Запуск:** Colab → загрузить этот `.ipynb` → выбрать GPU → загрузить JSONL;
Kaggle → Import Notebook → включить GPU и Internet → добавить приватный Dataset с JSONL.
Один GPU достаточен. Второй T4 автоматически не используется. Начните с замороженного encoder.
При нехватке памяти уменьшите `BATCH_SIZE` до 1. Интернет нужен для pip и базового checkpoint.


## 1. Что реально делает бот

| Решение | Текущая реализация | Что обучается |
|---|---|---|
| Реагировать на обращение | `VoiceSession`, имя целиком, тот же канал, владелец окна, таймеры и отмена | Правила остаются; отдельная экспериментальная схема ниже |
| Команда управления | Laya `action`: skip, pause, resume, stop, volume_set/up/down, unknown | Основная задача этого ноутбука |
| Автоплей, повтор, очередь, голос | `modeCommand` в TypeScript | Не добавляем неподдерживаемые классы в runtime `action` |
| Название, исполнитель, ссылка | `extractMusicRequest` → YouTube → первый корректный результат | Не требуется знать все названия песен |
| Громкость | Число разбирается кодом, 1–150%; относительное изменение ±10 | Laya выбирает действие, не извлекает число |
| Голосовой ответ | Готовые подтверждения `confirmations.json`, с учётом успешности действия | Модель не сочиняет текст ответа |

`music_route`, `music_policy` и `music_rerank` в `voice_worker.py` и схема
`laya-muz-bot-ds-v3` — сохранённый старый контракт. Сейчас `resolveMusicRequest` их не вызывает.
`/play`, slash-команды и кнопки не должны превращаться в классы голосовой модели.
Обучение новых голов само по себе не меняет поведение бота.


## 2. Данные и ручная разметка

На компьютере с ботом включите `VOICE_NLI_LOG=1`, соберите команды **после имени и сигнала**.
Журнал `.runtime/nli/commands-*.jsonl` содержит подсказки модели, но `gold=null`.
Он не содержит фоновых разговоров и обращений до сигнала: по нему нельзя обучить решение «реагировать».
Не считайте `suggested_intent`, `model_decision`, `outcome=changed` правильной разметкой.

Создайте `reviewed.jsonl`, по одной проверенной записи:
```json
{"event_id":"ID из журнала","intent":"pause"}
```
Экспортируйте **в новый файл**, затем загрузите его в Colab/Kaggle:
```powershell
npm run export:nli -- reviewed.jsonl reviewed-nli.jsonl
```
Либо загрузите исходные журналы и `reviewed.jsonl` и укажите `REVIEWS_PATH`.
Есть и прямой формат `commands-reviewed.jsonl`:
```json
{"event_id":"e1","message":"поставь на паузу","intent":"pause","reviewed":true,"group_id":"pause-family-01","player":{"connected":true,"playing":true,"paused":false,"autoplay":false,"queue_length":1}}
```
`group_id` объединяет варианты одной реплики, запись одного сеанса/говорящего или шаблон генерации.
Ставьте общий ID для родственных примеров. Точные дубликаты исходного и приведённого текста
объединяются автоматически. Если экспорт не содержит группы, можно добавить её вручную.
Без такой группировки метрики могут быть завышены.

Для стартового корпуса `lia-starter-v1` задайте `ALLOW_AUTOMATIC_LABELS=True`.
При этом дополнительно загружаются `commands-synthetic.jsonl` и `bot-commands-labeled.jsonl`.
Их метки помечены `reviewed=false`, `rule_verified` и происхождением, а не выдаются за ручной gold.
Непроверенные исходные логи и очередь ручной проверки автоматически в обучение не попадают.
Синтетический test и автоматическая разметка не измеряют качество на реальной речи пользователей.

Размечайте **запрос**, а не успешность операции: pause остаётся pause даже если уже пауза;
no_op, сбой сети и отмена не означают unknown. Испорченный STT нельзя исправлять подсказкой в state:
неоднозначную расшифровку исключите или разметьте unknown. Сохраняйте опечатки STT, если смысл понятен.
«Громче на 50», отрицания, несколько команд, чужие устройства — важные отрицательные примеры.
`play` — поиск названия/исполнителя; на уровне runtime action он становится unknown.
Режимы autoplay/loop/voice и queue_clear обрабатываются кодом; в runtime action это тоже unknown.
Для contextual «включи музыку» нужен реальный `player.paused`; без контекста такие строки исключите.

Нужны разнообразные независимые семейства каждого класса, включая unknown и музыкальные запросы.
Минимум 4 семьи на размеченный класс — лишь технический минимум для четырёх выборок,
а не достаточный объём для качественного обучения. Начальный ориентир: сотни проверенных
примеров на основной класс, много разных негативов и отдельные реальные записи для test.
Журналы содержат речь участников: загружайте только согласованные, очищенные данные в приватный Dataset.


## 3. Параметры

`DATA_PATHS` — список файлов. При пустом списке ищутся только `reviewed-nli.jsonl` и
`commands-reviewed.jsonl` в `/content/data`, `/content`, `/kaggle/input`.
Для Colab можно выполнить отдельную ячейку `from google.colab import files; files.upload()`.
`SMOKE_TEST=True` только проверяет код на крошечном искусственном наборе и помечает экспорт **не для бота**.
Используйте обычный режим с вашими метками для настоящего обучения.


In [ ]:
from pathlib import Path
import os
DATA_PATHS = []
REVIEWS_PATH = None
PLANNER_PATHS = []  # Опциональные вручную размеченные контекстные сценарии, раздел 7.
ALLOW_AUTOMATIC_LABELS = False  # Явное разрешение rule_verified: синтетика / автоматическая разметка логов.
AUTO_LOAD_PLANNER = False  # Искать planner-synthetic.jsonl рядом с выбранными файлами команд.
BASE_MODEL = "convaiinnovations/laya-multilingual"
BASE_REVISION = "e4e9ddf21a7b1903b7acffd8814ad4307bf63a67"  # Подготовленная версия этого бота.
SEED = 42
EPOCHS = 4
BATCH_SIZE = 2
ACCUMULATION = 8
HEAD_LR = 2e-5
ENCODER_LR = 2e-6
TRAIN_ENCODER = False
SMOKE_TEST = False
WORKDIR = Path("/kaggle/working/lia-training" if Path("/kaggle/working").exists() else "/content/lia-training")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("HF_HOME", str(WORKDIR / "hf-cache"))


## 4. Зависимости

Версия Laya соответствует `scripts/voice-requirements.txt`. PyTorch платформы сохраняется:
не устанавливаем CPU wheel поверх CUDA. После установки, если среда просит, перезапустите kernel.
Внутренние функции Laya используются с фиксированной версией; при обновлении нужна повторная проверка.
Метод обучения: supervised cross entropy по логитам вариантов. Это осознанно более простой
метод для ручных одношаговых меток, **не RLCD/GRPO**. Температуры подбираются отдельно.


In [ ]:
import subprocess, sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "laya==0.3.22", "transformers==5.17.0",
                "huggingface-hub==1.33.0", "matplotlib>=3.9,<4"], check=True)
if shutil.which("node") is None:
    if not shutil.which("apt-get"):
        raise RuntimeError("Install Node.js to run the exact TypeScript parser")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs"], check=True)


In [ ]:
import importlib.util, importlib.metadata, json, random, platform, shutil
import numpy as np
import torch
import laya
assert importlib.metadata.version("laya") == "0.3.22"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not SMOKE_TEST:
    raise RuntimeError("Select GPU in Colab/Kaggle before real training")
print({"device": DEVICE, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
       "torch": torch.__version__, "python": platform.python_version()})


## 5. Встроенный контракт бота

Следующая ячейка разворачивает снимок исходников. Расшифровки проходят **те же** TypeScript
`contextualCommand`, `modeCommand`, `extractMusicRequest`, `unsupportedSpeech`, `validateIntent`.
Мы не обучаем модель на JSON с полем intent, когда в production ей приходит строка и вопрос action.
`canonical_message` из журнала игнорируется и вычисляется заново из исходной речи и player.


In [ ]:
SNAPSHOT = json.loads('{"helper": "\\"\\"\\"Portable Laya training utilities, embedded in the Colab/Kaggle notebook.\\n\\nData preparation uses only the standard library. Torch/Laya imports are lazy.\\nTraining uses supervised option-level cross entropy, not text generation or RLCD.\\n\\"\\"\\"\\nfrom collections import Counter, defaultdict\\nfrom contextlib import nullcontext\\nfrom copy import deepcopy\\nimport hashlib\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport random\\nimport shutil\\nimport subprocess\\nimport unicodedata\\n\\n\\nSPLITS = (\\"train\\", \\"validation\\", \\"calibration\\", \\"test\\")\\nMODEL_KEYS = (\\"input_ids\\", \\"attention_mask\\", \\"marker_pos\\", \\"marker_mask\\", \\"qtype\\")\\n\\n\\ndef normalized(text):\\n    text = unicodedata.normalize(\\"NFKC\\", text).lower().replace(\\"ё\\", \\"е\\")\\n    return \\" \\".join(\\"\\".join(c if c.isalnum() else \\" \\" for c in text).split())\\n\\n\\ndef json_object(value):\\n    value = json.loads(value) if isinstance(value, str) else value\\n    if not isinstance(value, dict):\\n        raise ValueError(\\"Expected JSON object\\")\\n    return value\\n\\n\\ndef read_jsonl(paths):\\n    rows = []\\n    for path in paths:\\n        for line_number, line in enumerate(Path(path).read_text(encoding=\\"utf-8-sig\\").splitlines(), 1):\\n            if not line.strip():\\n                continue\\n            try:\\n                rows.append(json_object(line))\\n            except (ValueError, TypeError) as error:\\n                raise ValueError(f\\"Invalid JSONL at {Path(path).name}:{line_number}\\") from error\\n    return rows\\n\\n\\ndef automatic_label_allowed(row, allow_automatic_labels):\\n    return (allow_automatic_labels and row.get(\\"reviewed\\") is False\\n            and row.get(\\"label_status\\") == \\"rule_verified\\"\\n            and ((row.get(\\"origin\\") == \\"synthetic\\" and row.get(\\"label_source\\") == \\"synthetic_template_checked\\")\\n                 or (row.get(\\"origin\\") == \\"bot_log\\" and row.get(\\"label_source\\") == \\"bot_transcript_rules\\")))\\n\\n\\ndef load_commands(paths, reviews_path=None, intent_labels=(), allow_automatic_labels=False):\\n    \\"\\"\\"Accept the existing export, reviewed command rows, or logs + separate reviews.\\"\\"\\"\\n    reviews = {}\\n    if reviews_path:\\n        for row in read_jsonl([reviews_path]):\\n            key = row.get(\\"event_id\\")\\n            if not isinstance(key, str) or not key or key in reviews:\\n                raise ValueError(\\"Invalid/duplicate review event_id\\")\\n            reviews[key] = row[\\"intent\\"]\\n    found, seen, commands = set(), set(), []\\n    for row in read_jsonl(paths):\\n        exported = \\"human-reviewed\\" in row.get(\\"tags\\", []) and \\"source_event_id\\" in row\\n        key = row.get(\\"source_event_id\\") if exported else row.get(\\"event_id\\")\\n        if not isinstance(key, str) or not key or key in seen:\\n            raise ValueError(\\"Every source row needs a unique event_id/source_event_id\\")\\n        seen.add(key)\\n        if key in reviews:\\n            intent = reviews[key]\\n            found.add(key)\\n        elif exported:\\n            intent = json_object(row.get(\\"gold\\"))[\\"intent\\"]\\n        elif row.get(\\"reviewed\\") is True and row.get(\\"label_status\\") != \\"unreviewed\\":\\n            intent = row.get(\\"intent\\")\\n        elif automatic_label_allowed(row, allow_automatic_labels):\\n            intent = row.get(\\"intent\\")\\n        else:\\n            continue  # Predictions, outcomes and suggested_intent are never gold.\\n        if intent not in intent_labels:\\n            raise ValueError(f\\"Unknown reviewed intent for {key}\\")\\n        state = json_object(row[\\"state\\"]) if \\"state\\" in row else {\\"message\\": row.get(\\"message\\"), \\"player\\": row.get(\\"player\\")}\\n        message = state.get(\\"message\\")\\n        if not isinstance(message, str) or not message.strip() or len(message) > 1000:\\n            raise ValueError(f\\"Invalid message for {key}\\")\\n        # No canonical_message, model_decision, diagnostics, outcome or gold hints.\\n        player = state.get(\\"player\\")\\n        if player is not None:\\n            if not isinstance(player, dict) or any(type(player.get(k)) is not bool for k in (\\"connected\\", \\"playing\\", \\"paused\\", \\"autoplay\\")):\\n                raise ValueError(f\\"Invalid player state for {key}\\")\\n            if type(player.get(\\"queue_length\\")) is not int or player[\\"queue_length\\"] < 0:\\n                raise ValueError(f\\"Invalid queue_length for {key}\\")\\n        group = row.get(\\"group_id\\", key)\\n        if not isinstance(group, str) or not group:\\n            raise ValueError(\\"group_id must be a non-empty string\\")\\n        commands.append({\\"event_id\\": key, \\"message\\": message, \\"player\\": player,\\n                         \\"intent\\": intent, \\"group_id\\": group, \\"origin\\": row.get(\\"origin\\", \\"human-reviewed\\"),\\n                         \\"label_source\\": row.get(\\"label_source\\", \\"human-review\\")})\\n    if set(reviews) != found:\\n        raise ValueError(\\"Some reviewed events were not found in source files\\")\\n    if not commands:\\n        raise ValueError(\\"No manually reviewed commands; label/export data first, or explicitly allow rule-verified automatic labels\\")\\n    return commands\\n\\n\\ndef node_bridge(rows, script):\\n    executable = shutil.which(\\"node\\")\\n    if not executable:\\n        raise RuntimeError(\\"Node.js required for exact bot preprocessing; install nodejs and rerun\\")\\n    result = subprocess.run([executable, str(script)], input=json.dumps(rows, ensure_ascii=False),\\n                            text=True, encoding=\\"utf-8\\", capture_output=True, check=True)\\n    return json.loads(result.stdout)\\n\\n\\ndef prepare_commands(commands, bridge_script):\\n    processed = node_bridge(commands, bridge_script)\\n    return [{**row, **processed[i]} for i, row in enumerate(commands)]\\n\\n\\ndef grouped_split(commands, seed=42, fractions=(0.6, 0.15, 0.1, 0.15), min_groups_per_label=4):\\n    \\"\\"\\"Union raw/canonical duplicates and explicit families before stratification.\\n\\n    Context variants of the same phrase stay together, even with different gold.\\n    Search over deterministic candidate assignments; fail rather than silently\\n    lose a class in one of the four held-out partitions.\\n    \\"\\"\\"\\n    if len(fractions) != 4 or any(x <= 0 for x in fractions) or not math.isclose(sum(fractions), 1):\\n        raise ValueError(\\"Four positive split fractions must sum to 1\\")\\n    parent = list(range(len(commands)))\\n    def find(i):\\n        while parent[i] != i:\\n            parent[i] = parent[parent[i]]\\n            i = parent[i]\\n        return i\\n    def union(a, b):\\n        parent[find(a)] = find(b)\\n    owners = {}\\n    for i, row in enumerate(commands):\\n        keys = [(\\"family\\", row[\\"group_id\\"])]\\n        keys += [(\\"text\\", normalized(row[k])) for k in (\\"message\\", \\"canonical\\") if row.get(k)]\\n        for key in keys:\\n            if key in owners:\\n                union(i, owners[key])\\n            owners[key] = i\\n    groups = defaultdict(list)\\n    for i, row in enumerate(commands):\\n        groups[find(i)].append(row)\\n    label_groups = Counter(label for group in groups.values() for label in {r[\\"intent\\"] for r in group})\\n    scarce = {label: n for label, n in label_groups.items() if n < min_groups_per_label}\\n    if scarce:\\n        raise ValueError(f\\"Too few independent phrase families (need {min_groups_per_label}): {scarce}\\")\\n    labels = sorted(label_groups)\\n    totals = Counter(r[\\"intent\\"] for r in commands)\\n    rng, best, best_score = random.Random(seed), None, float(\\"inf\\")\\n    # Group assignment is chosen only from labels and sizes, never model results.\\n    for _ in range(1000):\\n        bins = {name: [] for name in SPLITS}\\n        for group in groups.values():\\n            name = rng.choices(SPLITS, weights=fractions)[0]\\n            bins[name].extend(group)\\n        counts = {s: Counter(r[\\"intent\\"] for r in bins[s]) for s in SPLITS}\\n        if any(not counts[s][label] for s in SPLITS for label in labels):\\n            continue\\n        score = sum(abs(counts[s][label] / totals[label] - fractions[j])\\n                    for j, s in enumerate(SPLITS) for label in labels)\\n        if score < best_score:\\n            best, best_score = bins, score\\n    if best is None:\\n        # Deterministic greedy coverage fallback for small homogeneous datasets.\\n        bins = {s: [] for s in SPLITS}\\n        counts = {s: Counter() for s in SPLITS}\\n        ordered = list(groups.values())\\n        rng.shuffle(ordered)\\n        ordered.sort(key=lambda g: min(label_groups[r[\\"intent\\"]] for r in g))\\n        for group in ordered:\\n            gc = Counter(r[\\"intent\\"] for r in group)\\n            def cost(s):\\n                j = SPLITS.index(s)\\n                uncovered = sum(counts[s][label] == 0 for label in gc)\\n                fill = sum(counts[s][label] / max(1, totals[label] * fractions[j]) for label in gc)\\n                return -1000 * uncovered + fill\\n            target = min(SPLITS, key=cost)\\n            bins[target].extend(group)\\n            counts[target].update(gc)\\n        if any(not counts[s][label] for s in SPLITS for label in labels):\\n            raise ValueError(\\"Cannot split these overlapping groups with class coverage; add independent families\\")\\n        best = bins\\n    return best\\n\\n\\ndef runtime_records(commands, worker):\\n    records = []\\n    allowed = worker.QUESTIONS[\\"action\\"][\\"criteria\\"]\\n    for command in commands:\\n        text = command[\\"canonical\\"].strip().rstrip(\'.!…\').strip().lower()\\n        text = text[:1].upper() + text[1:]\\n        # Both production prompts learn the same target; they share a split.\\n        target = command[\\"intent\\"] if command[\\"intent\\"] in allowed else \\"unknown\\"\\n        for prompt, questions in ((\\"primary\\", worker.QUESTIONS), (\\"short\\", worker.SHORT_QUESTIONS)):\\n            records.append({\\"event_id\\": command[\\"event_id\\"], \\"task\\": \\"action_\\" + prompt,\\n                            \\"state\\": text, \\"questions\\": questions, \\"gold\\": {\\"action\\": target}})\\n    return records\\n\\n\\ndef planner_questions(intent_questions):\\n    return {\\n        \\"should_respond\\": {\\"type\\": \\"choice\\", \\"instructions\\": \\"Нужно ли реагировать на эту реплику в текущей голосовой сессии?\\",\\n                           \\"criteria\\": {\\"respond\\": \\"Обращение по имени в idle либо команда владельца в awaiting; пользователь в канале бота, голос включён\\",\\n                                        \\"ignore\\": \\"Фоновый разговор, другой пользователь, нет доступа, голос выключен или сессия занята\\"}},\\n        \\"next_tool\\": {\\"type\\": \\"choice\\", \\"instructions\\": \\"Какую функцию музыкального бота выбрать? Не выполняй команды без активной сессии владельца.\\",\\n                      \\"criteria\\": {\\"wake_ack\\": \\"Сигнал на отдельное имя активации, затем ждать команду\\",\\n                                   \\"youtube_music_search\\": \\"Поиск и включение музыки по тексту\\",\\n                                   \\"direct_youtube_video\\": \\"Конкретное видео YouTube по ссылке\\",\\n                                   \\"player_control\\": \\"Команда управления плеером или его режимами\\",\\n                                   \\"no_tool\\": \\"Игнорировать или сообщить о неподдерживаемой команде без вызова инструмента\\"}},\\n        \\"intent\\": deepcopy(intent_questions[\\"intent\\"]),\\n    }\\n\\n\\ndef load_planner(paths, questions, bridge_script, allow_automatic_labels=False):\\n    records = []\\n    required = {\\"message\\", \\"phase\\", \\"voice_enabled\\", \\"in_bot_channel\\", \\"is_owner\\", \\"wake_name\\", \\"player\\"}\\n    source_rows = read_jsonl(paths)\\n    contexts = [json_object(row[\\"state\\"]) for row in source_rows]\\n    wakes = node_bridge([{\\"message\\": state.get(\\"message\\", \\"\\"), \\"wake_name\\": state.get(\\"wake_name\\", \\"\\")}\\n                         for state in contexts], bridge_script)\\n    for row, wake in zip(source_rows, wakes, strict=True):\\n        if row.get(\\"reviewed\\") is not True and not automatic_label_allowed(row, allow_automatic_labels):\\n            raise ValueError(\\"Planner examples must be manually reviewed\\")\\n        state, gold = json_object(row[\\"state\\"]), json_object(row[\\"gold\\"])\\n        if set(state) != required or state[\\"phase\\"] not in {\\"idle\\", \\"awaiting\\", \\"busy\\", \\"disabled\\"}:\\n            raise ValueError(\\"Planner state must use the documented voice context, without answer hints\\")\\n        if not isinstance(state[\\"message\\"], str) or len(state[\\"message\\"]) > 1000 or not isinstance(state[\\"wake_name\\"], str):\\n            raise ValueError(\\"Invalid planner text\\")\\n        if any(type(state[k]) is not bool for k in (\\"voice_enabled\\", \\"in_bot_channel\\", \\"is_owner\\")):\\n            raise ValueError(\\"Planner context flags must be booleans\\")\\n        if set(gold) != set(questions) or any(gold[k] not in questions[k][\\"criteria\\"] for k in questions):\\n            raise ValueError(\\"Invalid planner gold\\")\\n        can_wake = state[\\"phase\\"] == \\"idle\\" and wake[\\"wake_match\\"]\\n        can_command = state[\\"phase\\"] == \\"awaiting\\" and state[\\"is_owner\\"]\\n        eligible = state[\\"voice_enabled\\"] and state[\\"in_bot_channel\\"] and (can_wake or can_command)\\n        if (gold[\\"should_respond\\"] == \\"respond\\") != eligible:\\n            raise ValueError(\\"Planner respond label contradicts explicit session context\\")\\n        tool, intent = gold[\\"next_tool\\"], gold[\\"intent\\"]\\n        if (not eligible and (tool != \\"no_tool\\" or intent != \\"unknown\\")\\n                or (can_wake and eligible and (tool != \\"wake_ack\\" or intent != \\"unknown\\"))\\n                or (tool == \\"wake_ack\\" and not can_wake)\\n                or (tool in {\\"youtube_music_search\\", \\"direct_youtube_video\\"} and intent != \\"play\\")\\n                or (tool == \\"player_control\\" and intent in {\\"play\\", \\"unknown\\"})\\n                or (tool == \\"no_tool\\" and intent != \\"unknown\\")):\\n            raise ValueError(\\"Inconsistent planner function/intent/context labels\\")\\n        key, group = row.get(\\"event_id\\"), row.get(\\"group_id\\")\\n        if not isinstance(key, str) or not key or not isinstance(group, str) or not group:\\n            raise ValueError(\\"Planner requires event_id and group_id\\")\\n        records.append({\\"event_id\\": key, \\"group_id\\": group, \\"message\\": state[\\"message\\"],\\n                        \\"canonical\\": state[\\"message\\"], \\"intent\\": \\"|\\".join(gold.values()),\\n                        \\"task\\": \\"planner\\", \\"state\\": state, \\"questions\\": questions, \\"gold\\": gold,\\n                        \\"origin\\": row.get(\\"origin\\", \\"human-reviewed\\"), \\"label_source\\": row.get(\\"label_source\\", \\"human-review\\")})\\n    return records\\n\\n\\ndef encode_records(agent, records):\\n    items = []\\n    for record in records:\\n        questions, gold = record[\\"questions\\"], record[\\"gold\\"]\\n        for qid, question in questions.items():\\n            agent._check_question(qid, question)\\n            internal = agent._to_internal(question)\\n            item = agent._encode_state(record[\\"state\\"], [qid], {qid: internal})[0]\\n            labels = list(question[\\"criteria\\"])\\n            if item[\\"state_stats\\"][\\"truncated\\"]:\\n                raise ValueError(\\"Input was truncated; increase token budget\\")\\n            if item[\\"options\\"][\\"options_distinct\\"] != item[\\"options\\"][\\"options\\"]:\\n                raise ValueError(\\"Option truncation made labels indistinguishable\\")\\n            item.update(label=labels.index(gold[qid]), labels=labels,\\n                        event_id=record[\\"event_id\\"], task=record[\\"task\\"], qid=qid)\\n            items.append(item)\\n    return items\\n\\n\\ndef batches(items, batch_size, shuffle=False, seed=42):\\n    from laya.common import collate_items\\n    order = list(range(len(items)))\\n    if shuffle:\\n        random.Random(seed).shuffle(order)\\n    for start in range(0, len(order), batch_size):\\n        selected = [items[i] for i in order[start:start + batch_size]]\\n        yield selected, collate_items([[item] for item in selected], batches.pad_id)\\n\\n\\ndef amp_context(agent):\\n    import torch\\n    return torch.autocast(\\"cuda\\", dtype=agent.dtype) if agent.device.type == \\"cuda\\" else nullcontext()\\n\\n\\ndef raw_logits(agent, items, batch_size=8):\\n    import torch\\n    agent.model.eval()\\n    rows = []\\n    with torch.no_grad():\\n        for selected, batch in batches(items, batch_size):\\n            with amp_context(agent):\\n                logits, _ = agent.model(**{k: batch[k].to(agent.device) for k in MODEL_KEYS})\\n            for item, z in zip(selected, logits, strict=True):\\n                rows.append({\\"item\\": item, \\"logits\\": z[:len(item[\\"labels\\"])].float().cpu().tolist()})\\n    return rows\\n\\n\\ndef evaluate_logits(rows, temperatures=None):\\n    import numpy as np\\n    from laya.common import temp_bucket, ece_score\\n    predictions, results = [], {}\\n    for row in rows:\\n        item = row[\\"item\\"]\\n        t = 1.0 if temperatures is None else temperatures[\\"temperature_by_options\\"].get(\\n            temp_bucket(item[\\"qtype\\"], len(item[\\"labels\\"])), temperatures[\\"temperature\\"][item[\\"qtype\\"]])\\n        z = np.asarray(row[\\"logits\\"], dtype=float) / t\\n        p = np.exp(z - z.max()); p /= p.sum()\\n        y, pred = item[\\"label\\"], int(p.argmax())\\n        predictions.append({\\"event_id\\": item[\\"event_id\\"], \\"task\\": item[\\"task\\"], \\"qid\\": item[\\"qid\\"],\\n                            \\"gold\\": item[\\"labels\\"][y], \\"prediction\\": item[\\"labels\\"][pred],\\n                            \\"confidence\\": float(p[pred]), \\"correct\\": y == pred,\\n                            \\"nll\\": float(-np.log(max(p[y], 1e-12))),\\n                            \\"brier\\": float(((p - np.eye(len(p))[y]) ** 2).sum())})\\n    groups = defaultdict(list)\\n    for row in predictions:\\n        groups[row[\\"task\\"] + \\"/\\" + row[\\"qid\\"]].append(row)\\n    for key, group in groups.items():\\n        labels = sorted({r[\\"gold\\"] for r in group} | {r[\\"prediction\\"] for r in group})\\n        per_class = {}\\n        for label in labels:\\n            tp = sum(r[\\"gold\\"] == label and r[\\"prediction\\"] == label for r in group)\\n            actual = sum(r[\\"gold\\"] == label for r in group)\\n            predicted = sum(r[\\"prediction\\"] == label for r in group)\\n            per_class[label] = {\\"support\\": actual, \\"precision\\": tp / predicted if predicted else 0.0,\\n                                \\"recall\\": tp / actual if actual else 0.0,\\n                                \\"f1\\": 2 * tp / (actual + predicted) if actual + predicted else 0.0}\\n        results[key] = {\\"n\\": len(group), \\"accuracy\\": sum(r[\\"correct\\"] for r in group) / len(group),\\n                       \\"macro_f1\\": sum(x[\\"f1\\"] for x in per_class.values()) / len(per_class),\\n                       \\"nll\\": sum(r[\\"nll\\"] for r in group) / len(group),\\n                       \\"brier\\": sum(r[\\"brier\\"] for r in group) / len(group),\\n                       \\"ece\\": ece_score(np.array([r[\\"confidence\\"] for r in group]), np.array([r[\\"correct\\"] for r in group])),\\n                       \\"per_class\\": per_class}\\n    return results, predictions\\n\\n\\ndef save_checkpoint(agent, directory, model_name, temperatures=None):\\n    import torch\\n    from safetensors.torch import save_file\\n    directory = Path(directory)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    config = deepcopy(agent.cfg)\\n    config[\\"model_name\\"] = model_name  # Never pretend to be the legacy music-v3 profile.\\n    config[\\"temperature\\"] = temperatures[\\"temperature\\"] if temperatures else [1.0, 1.0, 1.0]\\n    config[\\"temperature_by_options\\"] = temperatures[\\"temperature_by_options\\"] if temperatures else {}\\n    config.pop(\\"lang_temperatures\\", None)\\n    config[\\"training\\"] = {\\"method\\": \\"supervised_option_cross_entropy\\", \\"fine_tuned_from_checkpoint\\": True}\\n    (directory / \\"rl_agent_config.json\\").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    tensors = {k: v.detach().cpu().contiguous().clone() for k, v in agent.model.state_dict().items()}\\n    tensors[\\"temperature\\"] = torch.tensor(config[\\"temperature\\"], dtype=tensors[\\"temperature\\"].dtype)\\n    save_file(tensors, str(directory / \\"model.safetensors\\"))\\n    agent.model.encoder.config.save_pretrained(str(directory / \\"encoder\\"))\\n    agent.tok.save_pretrained(str(directory / \\"tokenizer\\"))\\n\\n\\ndef train(agent, train_items, validation_items, directory, epochs=4, batch_size=2,\\n          accumulation=8, head_lr=2e-5, encoder_lr=2e-6, train_encoder=False, seed=42):\\n    import torch\\n    import torch.nn.functional as F\\n    from safetensors.torch import load_file\\n    torch.manual_seed(seed)\\n    for name, parameter in agent.model.named_parameters():\\n        parameter.requires_grad_(not name.startswith(\\"act_head.\\") and (train_encoder or not name.startswith(\\"encoder.\\")))\\n    # The separate act/escalate head is not the bot\'s action classifier and has no gold here.\\n    head = [p for n, p in agent.model.named_parameters() if p.requires_grad and not n.startswith(\\"encoder.\\")]\\n    encoder = [p for n, p in agent.model.named_parameters() if p.requires_grad and n.startswith(\\"encoder.\\")]\\n    groups = [{\\"params\\": head, \\"lr\\": head_lr}]\\n    if encoder:\\n        groups.append({\\"params\\": encoder, \\"lr\\": encoder_lr})\\n        agent.model.encoder.gradient_checkpointing_enable()\\n    agent.model.head_checkpointing = True\\n    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)\\n    scaler = torch.amp.GradScaler(\\"cuda\\", enabled=agent.device.type == \\"cuda\\" and agent.dtype == torch.float16)\\n    micro_batches = math.ceil(len(train_items) / batch_size)\\n    updates_per_epoch = math.ceil(micro_batches / accumulation)\\n    total_updates = epochs * updates_per_epoch\\n    warmup = max(1, int(total_updates * 0.1))\\n    def schedule(step):\\n        if step < warmup:\\n            return (step + 1) / warmup\\n        return max(0.0, (total_updates - step) / max(1, total_updates - warmup))\\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\\n    best, history = float(\\"inf\\"), []\\n    for epoch in range(epochs):\\n        agent.model.train()\\n        if not train_encoder:\\n            agent.model.encoder.eval()  # Freeze encoder dropout as well as its parameters.\\n        optimizer.zero_grad(set_to_none=True)\\n        loss_sum, count, updates = 0.0, 0, 0\\n        for index, (selected, batch) in enumerate(batches(train_items, batch_size, True, seed + epoch)):\\n            # Correct sample denominator also for an incomplete last accumulation window.\\n            window_start = (index // accumulation) * accumulation * batch_size\\n            window_size = min(accumulation * batch_size, len(train_items) - window_start)\\n            with amp_context(agent):\\n                logits, _ = agent.model(**{k: batch[k].to(agent.device) for k in MODEL_KEYS})\\n                loss = F.cross_entropy(logits.float(), batch[\\"label\\"].to(agent.device), reduction=\\"sum\\")\\n            if not torch.isfinite(loss):\\n                raise RuntimeError(\\"Non-finite training loss; reduce LR/use fp32\\")\\n            scaler.scale(loss / window_size).backward()\\n            loss_sum += loss.item(); count += len(selected)\\n            if (index + 1) % accumulation == 0 or index + 1 == micro_batches:\\n                scaler.unscale_(optimizer)\\n                torch.nn.utils.clip_grad_norm_([p for p in agent.model.parameters() if p.requires_grad], 1.0)\\n                scale_before = scaler.get_scale()\\n                scaler.step(optimizer); scaler.update()\\n                if scaler.get_scale() >= scale_before:\\n                    scheduler.step(); updates += 1\\n                optimizer.zero_grad(set_to_none=True)\\n        validation, _ = evaluate_logits(raw_logits(agent, validation_items, batch_size))\\n        val_nll = sum(x[\\"nll\\"] * x[\\"n\\"] for x in validation.values()) / sum(x[\\"n\\"] for x in validation.values())\\n        history.append({\\"epoch\\": epoch + 1, \\"train_nll\\": loss_sum / count, \\"validation_nll\\": val_nll, \\"updates\\": updates})\\n        print(history[-1], flush=True)\\n        if val_nll < best:\\n            best = val_nll\\n            save_checkpoint(agent, directory, \\"laya-muz-bot-controls-v1\\")\\n    agent.model.load_state_dict(load_file(str(Path(directory) / \\"model.safetensors\\")), strict=True)\\n    agent.model.eval()\\n    return history\\n\\n\\ndef calibrate(rows):\\n    from laya.calibrate import fit_temperature_map\\n    records = []\\n    # Do not count primary and fallback prompts as two independent events.\\n    seen = set()\\n    for row in rows:\\n        item = row[\\"item\\"]\\n        key = (item[\\"event_id\\"], item[\\"qid\\"])\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        k = len(item[\\"labels\\"])\\n        target = [float(i == item[\\"label\\"]) for i in range(k)]\\n        records.append((item[\\"qtype\\"], row[\\"logits\\"], target, k))\\n    result = fit_temperature_map(records, compute_ece=False)\\n    return {\\"temperature\\": result[\\"temperature\\"], \\"temperature_by_options\\": result[\\"temperature_by_options\\"]}\\n\\n\\ndef runtime_evaluation(agent, commands, worker, bridge_script):\\n    predictions = [worker.classify(agent, row[\\"canonical\\"]) for row in commands]\\n    checked = node_bridge([{**row, \\"decision\\": decision} for row, decision in zip(commands, predictions, strict=True)], bridge_script)\\n    rows = [{\\"event_id\\": row[\\"event_id\\"], \\"gold\\": row[\\"intent\\"], \\"prediction\\": result[\\"final_action\\"],\\n             \\"model_required\\": row[\\"model_required\\"], \\"confidence\\": decision[\\"confidence\\"]}\\n            for row, result, decision in zip(commands, checked, predictions, strict=True)]\\n    negatives = [r for r in rows if r[\\"gold\\"] in {\\"unknown\\", \\"play\\"}]\\n    required = [r for r in rows if r[\\"model_required\\"]]\\n    false_controls = sum(r[\\"prediction\\"] not in {\\"unknown\\", \\"play\\"} for r in negatives)\\n    return {\\"n\\": len(rows), \\"pipeline_accuracy\\": sum(r[\\"gold\\"] == r[\\"prediction\\"] for r in rows) / len(rows),\\n            \\"model_required_n\\": len(required),\\n            \\"model_required_accuracy\\": sum(r[\\"gold\\"] == r[\\"prediction\\"] for r in required) / len(required) if required else None,\\n            \\"negative_n\\": len(negatives), \\"false_control_count\\": false_controls,\\n            \\"false_control_rate\\": false_controls / len(negatives) if negatives else None,\\n            \\"note\\": \\"Current text pipeline only; not an audio, Discord, cancellation or live YouTube test\\"}, rows\\n\\n\\ndef sha256_file(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n", "worker": "\\"\\"\\"JSONL worker. Audio stays in RAM; stdout is exclusively the IPC protocol.\\"\\"\\"\\nimport base64\\nfrom contextlib import redirect_stdout\\nimport json\\nimport os\\nimport re\\nfrom pathlib import Path\\nimport sys\\n\\nROOT = Path(__file__).resolve().parent.parent\\nRUNTIME = ROOT / \\".runtime\\" / \\"voice\\"\\nDEFAULT_LAYA_MODEL = \\"convaiinnovations/laya-multilingual\\"\\nos.environ.setdefault(\\"HF_HOME\\", str(RUNTIME / \\"cache\\"))\\nos.environ.setdefault(\\"USE_TF\\", \\"0\\")\\n\\n\\ndef laya_source(manifest, prepare=False):\\n    configured = os.environ.get(\\"LAYA_MODEL_PATH\\", \\"\\").strip()\\n    if configured:\\n        directory = Path(configured).expanduser()\\n        if not directory.is_absolute():\\n            directory = ROOT / directory\\n        directory = directory.resolve()\\n        if not directory.is_dir():\\n            raise FileNotFoundError(\\"Configured local Laya checkpoint is missing\\")\\n        return str(directory), None\\n    return (DEFAULT_LAYA_MODEL, None) if prepare else (manifest.get(\\"laya\\", DEFAULT_LAYA_MODEL), manifest.get(\\"laya_revision\\"))\\n\\nCOMMAND_PROMPT = (\\"Музыкальный плеер: включи, выключи, бесконечный режим, автоплей, повтор трека, \\"\\n                  \\"очисти очередь, голосовое управление, поставь на паузу, продолжи музыку, \\"\\n                  \\"следующий трек, останови музыку, громкость, сделай громче, сделай тише. \\"\\n                  \\"Linkin Park, Numb, live, remix, cover. Монеточка, Земфира, Кино, Сплин, Баста, Би-2.\\")\\n\\n# Give the short, ambiguous name competing spellings instead of telling the\\n# recognizer that it must hear the bot\'s name. Similar words remain non-wakes.\\nWAKE_VOCABULARY = {\\"бот\\": \\"Бот, вот, кот, год, рот, борт, порт.\\"}\\n\\nMUSIC_QUESTIONS = {\\n    \\"next_tool\\": {\\n        \\"type\\": \\"choice\\",\\n        \\"instructions\\": \\"Определи команду.\\",\\n        \\"criteria\\": {\\n            \\"search\\": \\"включить песню\\",\\n            \\"url\\": \\"включить ссылку YouTube\\",\\n            \\"control\\": \\"пауза, продолжить, пропустить, громкость\\",\\n            \\"unknown\\": \\"другая просьба\\",\\n        },\\n    },\\n    \\"search_result_policy\\": {\\n        \\"type\\": \\"choice\\",\\n        \\"instructions\\": \\"Какая версия песни нужна?\\",\\n        \\"criteria\\": {\\n            \\"original\\": \\"обычная песня\\",\\n            \\"live\\": \\"live концертная версия\\",\\n            \\"remix\\": \\"remix ремикс\\",\\n            \\"cover\\": \\"cover кавер\\",\\n            \\"acoustic\\": \\"acoustic акустическая\\",\\n            \\"instrumental\\": \\"instrumental инструментальная\\",\\n        },\\n    },\\n}\\n\\nMUSIC_SHORT_QUESTIONS = {\\"next_tool\\": {\\"type\\": \\"choice\\", \\"instructions\\": \\"Определи команду.\\", \\"criteria\\": {\\n    \\"search\\": \\"включить песню\\", \\"url\\": \\"включить ссылку YouTube\\",\\n    \\"control\\": \\"управлять плеером\\", \\"unknown\\": \\"другая просьба\\",\\n}}}\\n\\nVERSION_ALIASES = (\\n    (r\\"\\\\bконцертн\\\\w*(?:\\\\s+верси\\\\w*)?\\\\b\\", \\"live\\"),\\n    (r\\"\\\\bремикс\\\\w*\\\\b\\", \\"remix\\"),\\n    (r\\"\\\\bкавер\\\\w*\\\\b\\", \\"cover\\"),\\n    (r\\"\\\\bакустическ\\\\w*(?:\\\\s+верси\\\\w*)?\\\\b\\", \\"acoustic\\"),\\n    (r\\"\\\\bинструментальн\\\\w*(?:\\\\s+верси\\\\w*)?\\\\b\\", \\"instrumental\\"),\\n)\\n\\nV3_SCHEMA = json.loads((ROOT / \\"scripts\\" / \\"laya_music_v3.json\\").read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef is_music_v3(agent):\\n    return getattr(agent, \\"cfg\\", {}).get(\\"model_name\\") == V3_SCHEMA[\\"profile\\"]\\n\\n\\ndef requested_variant(query):\\n    for pattern, name in ((r\\"\\\\b(live|концертн\\\\w*)\\\\b\\", \\"live\\"), (r\\"\\\\b(remix|ремикс\\\\w*)\\\\b\\", \\"remix\\"),\\n                          (r\\"\\\\b(cover|кавер\\\\w*)\\\\b\\", \\"cover\\"), (r\\"\\\\b(acoustic|акустическ\\\\w*)\\\\b\\", \\"acoustic\\"),\\n                          (r\\"\\\\b(instrumental|инструментальн\\\\w*)\\\\b\\", \\"instrumental\\")):\\n        if re.search(pattern, query, re.IGNORECASE):\\n            return name\\n    return None\\n\\n\\ndef model_music_query(query):\\n    for pattern, version in VERSION_ALIASES:\\n        query = re.sub(pattern, version, query, flags=re.IGNORECASE)\\n    query = re.sub(r\\"\\\\b(live|remix|cover|acoustic|instrumental)\\\\b\\", lambda match: match.group().lower(), query, flags=re.IGNORECASE)\\n    return query\\n\\n\\ndef validate_candidates(query, candidates):\\n    if not isinstance(query, str) or not query or len(query) > 1000 or not isinstance(candidates, list) or not 1 <= len(candidates) <= 5:\\n        raise ValueError(\\"Invalid candidates\\")\\n    for index, item in enumerate(candidates):\\n        if not isinstance(item, dict) or item.get(\\"index\\") != index:\\n            raise ValueError(\\"Invalid index\\")\\n        for key, limit in ((\\"title\\", 240), (\\"artist\\", 120), (\\"duration\\", 32)):\\n            if not isinstance(item.get(key), str) or len(item[key]) > limit:\\n                raise ValueError(\\"Invalid candidate\\")\\n\\n\\ndef v3_candidates(candidates):\\n    results = []\\n    for item in candidates:\\n        duration = item[\\"duration\\"].split(\\":\\")\\n        seconds = 0\\n        if all(part.isdigit() for part in duration):\\n            for part in duration:\\n                seconds = seconds * 60 + int(part)\\n        title = item[\\"title\\"]\\n        for pattern, version in VERSION_ALIASES:\\n            title = re.sub(pattern, version, title, flags=re.IGNORECASE)\\n        results.append({\\"id\\": f\'c{item[\\"index\\"]}\', \\"title\\": title, \\"artist\\": item[\\"artist\\"], \\"duration\\": seconds})\\n    return results\\n\\n\\ndef v3_music_route(agent, state):\\n    parsed = state.get(\\"parsed_request\\")\\n    if not isinstance(parsed, dict) or parsed.get(\\"kind\\") != \\"play\\":\\n        raise ValueError(\\"Missing parsed music request\\")\\n    query = parsed.get(\\"search_query\\")\\n    if query is not None and (not isinstance(query, str) or not query or len(query) > 1000):\\n        raise ValueError(\\"Invalid parsed query\\")\\n    model_state = {\\"phase\\": \\"request\\", \\"message\\": state[\\"message\\"], \\"parsed_request\\": parsed,\\n                   \\"selected_track\\": None, \\"current_track\\": None,\\n                   \\"player\\": state.get(\\"player\\", {\\"connected\\": True, \\"playing\\": False, \\"paused\\": False,\\n                                                  \\"autoplay\\": False, \\"queue_length\\": 0}),\\n                   \\"available_tools\\": [\\"youtube_music_search\\", \\"direct_youtube_video\\", \\"player_control\\"]}\\n    answers = agent.predict(model_state, V3_SCHEMA[\\"routing\\"], max_len=2048, head_max_len=768)[\\"answers\\"]\\n    route = answers[\\"next_tool\\"]\\n    tool = route[\\"choice\\"] if route[\\"choice\\"] in model_state[\\"available_tools\\"] else \\"unknown\\"\\n    confidence = route[\\"answer_confidence\\"]\\n    if tool == \\"youtube_music_search\\":\\n        source = answers[\\"query_source\\"]\\n        confidence = min(confidence, source[\\"answer_confidence\\"])\\n        if source[\\"choice\\"] != \\"parsed_search_query\\":\\n            tool = \\"unknown\\"\\n    return {\\"next_tool\\": tool, \\"query_source\\": \\"message\\", \\"search_result_policy\\": \\"play_first_result\\",\\n            \\"defer_result_policy\\": tool == \\"youtube_music_search\\", \\"confidence\\": confidence}\\n\\n\\ndef music_result_policy(agent, query, candidates):\\n    validate_candidates(query, candidates)\\n    if not is_music_v3(agent):\\n        raise ValueError(\\"Result-policy phase is only used by the v3 profile\\")\\n    state = {\\"phase\\": \\"tool_result\\", \\"tool\\": \\"youtube_music_search\\", \\"query\\": model_music_query(query),\\n             \\"request_variant\\": requested_variant(query), \\"results\\": v3_candidates(candidates)}\\n    answer = agent.predict(state, V3_SCHEMA[\\"tool_result\\"], max_len=2048, head_max_len=768)[\\"answers\\"][\\"search_result_policy\\"]\\n    return {\\"search_result_policy\\": answer[\\"choice\\"], \\"confidence\\": answer[\\"answer_confidence\\"]}\\n\\n\\ndef v3_music_rerank(agent, query, candidates):\\n    items = v3_candidates(candidates)\\n    version = requested_variant(query)\\n    confidence = {}\\n    if version:\\n        # A trained best_track head can be confident about an incompatible\\n        # version. Confirm candidates\' versions independently, as in the\\n        # multilingual adapter, before comparing otherwise compatible tracks.\\n        decisions = agent.predict_batch(candidates, {\\"version\\": MUSIC_QUESTIONS[\\"search_result_policy\\"]},\\n                                        batch_size=5, max_len=2048, head_max_len=768)\\n        for item, decision in zip(items, decisions, strict=True):\\n            answer = decision[\\"answers\\"][\\"version\\"]\\n            probability = answer[\\"probabilities\\"].get(version, 0)\\n            if answer[\\"choice\\"] == version and probability >= 0.6:\\n                confidence[item[\\"id\\"]] = probability\\n        items = [item for item in items if item[\\"id\\"] in confidence]\\n        if not items:\\n            return {\\"best_track\\": 0, \\"confidence\\": 0}\\n    # Metadata is already in the candidate state. Keep the option text to the\\n    # title, rather than changing the trained choice shape with extra suffixes.\\n    criteria = {item[\\"id\\"]: item[\\"title\\"] for item in items}\\n    criteria[\\"none\\"] = V3_SCHEMA[\\"rerank\\"][\\"none\\"]\\n    questions = {\\"best_track\\": {\\"type\\": \\"choice\\", \\"instructions\\": V3_SCHEMA[\\"rerank\\"][\\"instructions\\"], \\"criteria\\": criteria}}\\n    state = {\\"phase\\": \\"rerank\\", \\"query\\": model_music_query(query), \\"constraints\\": {\\"variant\\": version}, \\"candidates\\": items}\\n    answer = agent.predict(state, questions, max_len=2048, head_max_len=768)[\\"answers\\"][\\"best_track\\"]\\n    choice = answer[\\"choice\\"]\\n    if choice == \\"none\\":\\n        return {\\"best_track\\": -1, \\"no_match\\": True, \\"confidence\\": answer[\\"answer_confidence\\"]}\\n    if choice not in {item[\\"id\\"] for item in items}:\\n        raise ValueError(\\"Invalid model candidate\\")\\n    return {\\"best_track\\": int(choice[1:]), \\"confidence\\": min(answer[\\"answer_confidence\\"], confidence.get(choice, 1))}\\n\\n\\ndef music_route(agent, state):\\n    if not isinstance(state, dict) or not isinstance(state.get(\\"message\\"), str) or len(state[\\"message\\"]) > 1000 or state.get(\\"selected_track\\") is not None:\\n        raise ValueError(\\"Invalid state\\")\\n    if is_music_v3(agent):\\n        return v3_music_route(agent, state)\\n    # Canonicalize explicit version aliases for the decision model only. The\\n    # original transcript/query remains in TypeScript and is sent to yt-dlp.\\n    state = {\\"message\\": state[\\"message\\"], \\"selected_track\\": None}\\n    for pattern, version in VERSION_ALIASES:\\n        state[\\"message\\"] = re.sub(pattern, version, state[\\"message\\"], flags=re.IGNORECASE)\\n    answers = agent.predict(state, MUSIC_QUESTIONS, max_len=2048, head_max_len=768)[\\"answers\\"]\\n    route = answers[\\"next_tool\\"]\\n    if route[\\"choice\\"] != \\"unknown\\" and route[\\"answer_confidence\\"] < 0.6:\\n        route = agent.predict(state, MUSIC_SHORT_QUESTIONS, max_len=2048, head_max_len=768)[\\"answers\\"][\\"next_tool\\"]\\n    policy = answers[\\"search_result_policy\\"]\\n    tools = {\\"search\\": \\"youtube_music_search\\", \\"url\\": \\"direct_youtube_video\\", \\"control\\": \\"player_control\\", \\"unknown\\": \\"unknown\\"}\\n    return {\\"next_tool\\": tools[route[\\"choice\\"]], \\"query_source\\": \\"message\\",\\n            \\"search_result_policy\\": \\"rerank_results\\" if policy[\\"choice\\"] != \\"original\\" and policy[\\"answer_confidence\\"] >= 0.6 else \\"play_first_result\\",\\n            \\"confidence\\": route[\\"answer_confidence\\"]}\\n\\n\\ndef music_rerank(agent, query, candidates):\\n    validate_candidates(query, candidates)\\n    if is_music_v3(agent):\\n        return v3_music_rerank(agent, query, candidates)\\n    version = requested_variant(query)\\n    # First let Laya identify each candidate\'s actual version, without the\\n    # requested version in that state (it otherwise mistakes query text for\\n    # candidate metadata). Compare only model-confirmed versions in pass two.\\n    confidence = {}\\n    if version:\\n        results = agent.predict_batch(candidates, {\\"version\\": MUSIC_QUESTIONS[\\"search_result_policy\\"]},\\n                                      batch_size=5, max_len=2048, head_max_len=768)\\n        for item, result in zip(candidates, results, strict=True):\\n            answer = result[\\"answers\\"][\\"version\\"]\\n            probability = answer[\\"probabilities\\"].get(version, 0)\\n            if answer[\\"choice\\"] == version and probability >= 0.6:\\n                confidence[item[\\"index\\"]] = probability\\n        candidates = [item for item in candidates if item[\\"index\\"] in confidence]\\n        if not candidates:\\n            return {\\"best_track\\": 0, \\"confidence\\": 0}\\n        if len(candidates) == 1:\\n            index = candidates[0][\\"index\\"]\\n            return {\\"best_track\\": index, \\"confidence\\": confidence[index]}\\n    criteria = {str(item[\\"index\\"]): f\'{item[\\"title\\"]} ({item[\\"artist\\"]}; {item[\\"duration\\"]})\' for item in candidates}\\n    questions = {\\"best_track\\": {\\"type\\": \\"choice\\",\\n        \\"instructions\\": \\"Выбери песню.\\",\\n        \\"criteria\\": criteria}}\\n    answer = agent.predict({\\"query\\": query}, questions, max_len=2048, head_max_len=768)[\\"answers\\"][\\"best_track\\"]\\n    index = int(answer[\\"choice\\"])\\n    return {\\"best_track\\": index, \\"confidence\\": min(answer[\\"answer_confidence\\"], confidence.get(index, 1))}\\n\\nQUESTIONS = {\\"action\\": {\\n    \\"type\\": \\"choice\\",\\n    \\"instructions\\": \\"Выбери ровно одно действие музыкального бота по просьбе пользователя. Непонятная, отрицательная, составная просьба или заказ музыки: unknown.\\",\\n    \\"criteria\\": {\\n        \\"skip\\": \\"Пропустить текущий трек, следующий трек\\",\\n        \\"pause\\": \\"Приостановить музыку, поставить на паузу\\",\\n        \\"resume\\": \\"Продолжить музыку после паузы\\",\\n        \\"stop\\": \\"Остановить музыку и отключить бота от голосового канала\\",\\n        \\"volume_set\\": \\"Установить громкость на указанное число процентов\\",\\n        \\"volume_up\\": \\"Сделать музыку громче без указания числа\\",\\n        \\"volume_down\\": \\"Сделать музыку тише без указания числа\\",\\n        \\"unknown\\": \\"Любая другая, неоднозначная или составная просьба; поиск или заказ песни\\",\\n    },\\n}}\\n\\n# A shorter alternative helps short commands whose first decision is uncertain.\\n# Both passes use the same loaded checkpoint. A weak second decision abstains.\\nSHORT_QUESTIONS = {\\"action\\": {\\"type\\": \\"choice\\", \\"instructions\\": \\"Определи команду.\\", \\"criteria\\": {\\n    \\"skip\\": \\"следующий трек\\", \\"pause\\": \\"поставить на паузу\\", \\"resume\\": \\"снять с паузы\\",\\n    \\"stop\\": \\"остановить музыку и выйти\\", \\"volume_set\\": \\"громкость в процентах\\",\\n    \\"volume_up\\": \\"сделать громче\\", \\"volume_down\\": \\"сделать тише\\", \\"unknown\\": \\"другая просьба\\",\\n}}}\\n\\n\\ndef classify(agent, text):\\n    # ASR casing/punctuation should not change the action of a short command.\\n    text = text.strip().rstrip(\'.!…\').strip().lower()\\n    text = text[:1].upper() + text[1:]\\n    if is_music_v3(agent) and text == \\"Сними с паузы\\":\\n        text = \\"Продолжи музыку\\"\\n    answer = agent.predict(text, QUESTIONS)[\\"answers\\"][\\"action\\"]\\n    if answer[\\"choice\\"] != \\"unknown\\" and answer[\\"answer_confidence\\"] < 0.6:\\n        answer = agent.predict(text, SHORT_QUESTIONS)[\\"answers\\"][\\"action\\"]\\n        if answer[\\"answer_confidence\\"] < 0.75:\\n            return {\\"action\\": \\"unknown\\", \\"confidence\\": answer[\\"answer_confidence\\"]}\\n    return {\\"action\\": answer[\\"choice\\"], \\"confidence\\": answer[\\"answer_confidence\\"]}\\n\\n\\ndef whisper_names(manifest, prepare=False):\\n    if prepare:\\n        return (os.environ.get(\\"WHISPER_MODEL\\", \\"large-v3-turbo\\"),\\n                os.environ.get(\\"WHISPER_WAKE_MODEL\\", \\"small\\"))\\n    # Old installations used one model for both phases.\\n    return manifest[\\"whisper\\"], manifest.get(\\"whisper_wake\\", manifest[\\"whisper\\"])\\n\\n\\ndef models(prepare=False):\\n    import torch\\n    # PyTorch\'s Windows wheel bundles the CUDA/cuDNN DLLs needed by CTranslate2.\\n    # Keep DLL-directory handles alive for the lifetime of the worker.\\n    dll_handles = []\\n    if os.name == \\"nt\\":\\n        torch_lib = str(Path(torch.__file__).parent / \\"lib\\")\\n        os.environ[\\"PATH\\"] = torch_lib + os.pathsep + os.environ.get(\\"PATH\\", \\"\\")\\n        dll_handles.append(os.add_dll_directory(torch_lib))\\n    import laya\\n    from faster_whisper import WhisperModel\\n    import numpy as np\\n\\n    class CommandWhisper(WhisperModel):\\n        # Keep standard context for command recognition: truncating the larger\\n        # encoder\'s window can make short utterances repeat. The small wake\\n        # model retains its separately tested low-latency window.\\n        encoder_frames = 3000\\n        encoder_cache = None\\n\\n        def detect_language(self, audio=None, features=None, **kwargs):\\n            # faster-whisper decodes shape[-1] - 1 content frames but language\\n            # detection includes the final boundary frame. Align that boundary\\n            # for a short command so both stages can reuse the exact encoding.\\n            if features is not None and 1 < features.shape[-1] < self.feature_extractor.nb_max_frames:\\n                features = features[..., :-1]\\n            return super().detect_language(audio=audio, features=features, **kwargs)\\n\\n        def encode(self, features):\\n            features = features[..., :self.encoder_frames]\\n            # Automatic language detection and decoding can encode the same\\n            # window twice. Reuse it only when the feature arrays match exactly.\\n            if self.encoder_cache is not None:\\n                previous, output = self.encoder_cache\\n                if np.array_equal(previous, features):\\n                    return output\\n            output = super().encode(features)\\n            self.encoder_cache = (features.copy(), output)\\n            return output\\n\\n        def command(self, audio, prompt=None, diagnostic=False):\\n            self.encoder_cache = None\\n            frames = int(len(audio) / 16000 * 100) + 150\\n            self.encoder_frames = 3000 if prompt is None else min(3000, max(600, (frames + 1) // 2 * 2))\\n            # Wake words keep the fast greedy pass. Song/artist names benefit\\n            # from comparing several hypotheses, without a fixed artist list.\\n            segments, info = self.transcribe(audio, language=None if prompt is None else \\"ru\\",\\n                                          task=\\"transcribe\\", beam_size=5 if prompt is None else 1, temperature=0,\\n                                          condition_on_previous_text=False, vad_filter=True,\\n                                          initial_prompt=COMMAND_PROMPT if prompt is None else prompt,\\n                                          without_timestamps=True, max_new_tokens=96)\\n            segments = list(segments)\\n            accepted = [segment for segment in segments\\n                        if prompt is None or (segment.no_speech_prob < 0.6 and segment.avg_logprob >= -1.0)]\\n            text = \\" \\".join(segment.text.strip() for segment in accepted)\\n            self.encoder_cache = None\\n            if diagnostic:\\n                return {\\"text\\": text, \\"metrics\\": {\\"vadMs\\": round(info.duration_after_vad * 1000),\\n                        \\"segments\\": len(segments), \\"rejectedSegments\\": len(segments) - len(accepted)}}\\n            return text\\n    torch.set_num_threads(max(1, min(8, int(os.environ.get(\\"VOICE_CPU_THREADS\\", \\"4\\")))))\\n    device = os.environ.get(\\"VOICE_DEVICE\\", \\"cpu\\").strip().lower()\\n    if device not in (\\"cpu\\", \\"cuda\\"):\\n        raise ValueError(\\"VOICE_DEVICE must be cpu or cuda\\")\\n    if device == \\"cuda\\" and not torch.cuda.is_available():\\n        raise RuntimeError(\\"CUDA requested but unavailable; install CUDA PyTorch or use VOICE_DEVICE=cpu\\")\\n    compute_type = \\"int8_float16\\" if device == \\"cuda\\" else \\"int8\\"\\n    print(f\\"Voice inference: {device}, Whisper {compute_type}\\", file=sys.stderr, flush=True)\\n    marker = RUNTIME / \\"ready.json\\"\\n    manifest = {} if prepare else json.loads(marker.read_text(encoding=\\"utf-8\\"))\\n    name, wake_name = whisper_names(manifest, prepare)\\n    def load_whisper(model_name):\\n        model = CommandWhisper(model_name, device=device, compute_type=compute_type, cpu_threads=torch.get_num_threads(),\\n                               download_root=str(RUNTIME / \\"whisper\\"), local_files_only=not prepare)\\n        model.runtime_name = model_name\\n        return model\\n    whisper = load_whisper(name)\\n    wake_whisper = whisper if name == wake_name else load_whisper(wake_name)\\n    laya_model, revision = laya_source(manifest, prepare)\\n    if prepare and not Path(laya_model).is_dir():\\n        from huggingface_hub import model_info\\n        revision = model_info(laya_model).sha\\n    agent = laya.load(laya_model, device=device, revision=revision)\\n    agent._voice_dll_handles = dll_handles\\n    print(f\\"Laya inference: {agent.device}\\", file=sys.stderr, flush=True)\\n    # Warm encoder/decoder and VAD as well as Laya before the first real wake.\\n    for model in (whisper,) if whisper is wake_whisper else (whisper, wake_whisper):\\n        model.encoder_frames = 3000 if model is whisper else 600\\n        warm_segments, _ = model.transcribe(np.zeros(16000, dtype=np.float32), language=\\"ru\\",\\n                                           beam_size=1, temperature=0, without_timestamps=True,\\n                                           max_new_tokens=8, condition_on_previous_text=False)\\n        list(warm_segments)\\n    whisper.command(np.zeros(16000, dtype=np.float32))\\n    wake_whisper.command(np.zeros(16000, dtype=np.float32), \\"\\")\\n    agent.predict(\\"Следующий трек\\", QUESTIONS)\\n    if prepare:\\n        temporary = marker.with_suffix(\'.json.tmp\')\\n        temporary.write_text(json.dumps({\\"whisper\\": name, \\"whisper_wake\\": wake_name,\\n                                        \\"laya\\": laya_model, \\"laya_revision\\": revision}), encoding=\\"utf-8\\")\\n        temporary.replace(marker)\\n    return whisper, wake_whisper, agent\\n\\n\\ndef main():\\n    prepare = \\"--prepare\\" in sys.argv\\n    with redirect_stdout(sys.stderr):\\n        whisper, wake_whisper, agent = models(prepare)\\n    if prepare:\\n        return\\n    print(json.dumps({\\"ready\\": True, \\"modelName\\": agent.cfg.get(\\"model_name\\", \\"laya-multilingual\\"),\\n                      \\"sttModelName\\": whisper.runtime_name,\\n                      \\"wakeModelName\\": wake_whisper.runtime_name,\\n                      \\"inferenceDevice\\": whisper.model.device}), flush=True)\\n    import numpy as np\\n    for line in sys.stdin:\\n        request = {}\\n        try:\\n            if len(line) > 700_000:\\n                raise ValueError(\\"Request too large\\")\\n            request = json.loads(line)\\n            with redirect_stdout(sys.stderr):\\n                if request[\\"op\\"] == \\"transcribe\\":\\n                    pcm = base64.b64decode(request[\\"pcm\\"], validate=True)\\n                    if not pcm or len(pcm) > 480_000 or len(pcm) % 2:\\n                        raise ValueError(\\"Invalid audio\\")\\n                    audio = np.frombuffer(pcm, dtype=\\"<i2\\").astype(np.float32) / 32768.0\\n                    wake_name = request.get(\\"wakeName\\")\\n                    # A single-name instruction biases noise toward that name.\\n                    # A contrastive vocabulary helps distinguish short words.\\n                    prompt = WAKE_VOCABULARY.get(wake_name.strip().lower(), \\"\\") if isinstance(wake_name, str) and len(wake_name) <= 64 else None\\n                    model = whisper if prompt is None else wake_whisper\\n                    result = model.command(audio, prompt, diagnostic=True)\\n                elif request[\\"op\\"] == \\"classify\\":\\n                    text = request[\\"text\\"]\\n                    if not isinstance(text, str) or len(text) > 1000:\\n                        raise ValueError(\\"Invalid text\\")\\n                    result = classify(agent, text)\\n                elif request[\\"op\\"] == \\"music_route\\":\\n                    result = music_route(agent, request[\\"state\\"])\\n                elif request[\\"op\\"] == \\"music_rerank\\":\\n                    result = music_rerank(agent, request[\\"query\\"], request[\\"candidates\\"])\\n                elif request[\\"op\\"] == \\"music_policy\\":\\n                    result = music_result_policy(agent, request[\\"query\\"], request[\\"candidates\\"])\\n                else:\\n                    raise ValueError(\\"Unknown operation\\")\\n            print(json.dumps({\\"id\\": request[\\"id\\"], \\"result\\": result}, ensure_ascii=False), flush=True)\\n        except Exception:\\n            # No exception details: they can contain audio, text or prompts.\\n            print(json.dumps({\\"id\\": request.get(\\"id\\"), \\"error\\": True}), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "music_schema": {"profile": "laya-muz-bot-ds-v3", "contract_source": "User-provided v3 state and option labels; question wording verified with the local checkpoint", "routing": {"next_tool": {"type": "choice", "instructions": "Какой инструмент нужно вызвать следующим?", "criteria": {"youtube_music_search": "Поиск музыки по текстовому запросу", "direct_youtube_video": "Воспроизведение конкретного видео YouTube по ссылке", "youtube_playlist": "Получение плейлиста YouTube по ссылке", "player_control": "Управление музыкальным плеером", "no_tool": "Не вызывать инструмент"}}, "query_source": {"type": "choice", "instructions": "Откуда взять поисковый запрос?", "criteria": {"parsed_search_query": "Поисковый запрос из parsed_request.search_query", "selected_track": "Выбранный пользователем трек", "current_track": "Текущий воспроизводимый трек", "none": "Поисковый запрос не нужен"}}}, "tool_result": {"search_result_policy": {"type": "choice", "instructions": "Как обработать результаты поиска?", "criteria": {"play_first_result": "Воспроизвести первый результат поиска", "rerank_results": "Сравнить результаты и выбрать лучшую версию", "no_result": "Нет подходящих результатов"}}}, "rerank": {"instructions": "Какой кандидат лучше всего соответствует запросу?", "none": "Ни один кандидат не подходит"}}, "intent_questions": {"intent": {"type": "choice", "instructions": "Определи одно действие музыкального бота. Непонятная, отрицательная или составная просьба: unknown.", "criteria": {"play": "Найти и включить песню или исполнителя", "skip": "Пропустить трек, включить следующий", "pause": "Поставить на паузу", "resume": "Продолжить после паузы", "stop": "Остановить и отключиться", "volume_set": "Установить громкость в процентах", "volume_up": "Сделать громче", "volume_down": "Сделать тише", "autoplay_on": "Включить рекомендации YouTube", "autoplay_off": "Выключить рекомендации YouTube", "loop_on": "Повторять текущий трек", "loop_off": "Выключить повтор трека", "queue_clear": "Очистить очередь", "voice_on": "Включить голосовое управление", "voice_off": "Выключить голосовое управление", "unknown": "Другая, отрицательная или составная просьба"}}}, "bridge": "function normalizeSpeech(value) {\\n    return value.normalize(\'NFKC\').toLocaleLowerCase(\'ru\').replace(/ё/g, \'е\')\\n        .replace(/[^\\\\p{L}\\\\p{N}]+/gu, \' \').trim().replace(/\\\\s+/g, \' \');\\n}\\nfunction validateWakeName(value) {\\n    const name = value.trim();\\n    if (name.length > 64 || normalizeSpeech(name).length < 2) {\\n        throw new Error(\'Имя должно содержать от 2 до 64 символов, включая буквы или цифры.\');\\n    }\\n    return name;\\n}\\nfunction isWakePhrase(text, name) {\\n    const key = wakeSpelling(name);\\n    return key.length >= 2 && wakeSpelling(text) === key;\\n}\\n// Whisper can spell a Russian name in Latin letters. Accept equivalent\\n// spelling without broadening the match to similar-sounding Russian words.\\nconst LATIN = {\\n    а: \'a\', б: \'b\', в: \'v\', г: \'g\', д: \'d\', е: \'e\', ж: \'zh\', з: \'z\', и: \'i\', й: \'y\',\\n    к: \'k\', л: \'l\', м: \'m\', н: \'n\', о: \'o\', п: \'p\', р: \'r\', с: \'s\', т: \'t\', у: \'u\',\\n    ф: \'f\', х: \'kh\', ц: \'ts\', ч: \'ch\', ш: \'sh\', щ: \'shch\', ъ: \'\', ы: \'y\', ь: \'\', э: \'e\', ю: \'yu\', я: \'ya\',\\n};\\nfunction wakeSpelling(value) {\\n    return normalizeSpeech(value).replace(/[а-я]/g, (letter) => LATIN[letter]);\\n}\\nfunction wakeDistance(text, name) {\\n    const actual = wakeSpelling(text).slice(0, 128), expected = wakeSpelling(name);\\n    let previous = Array.from({ length: expected.length + 1 }, (_, i) => i);\\n    for (let i = 0; i < actual.length; i++) {\\n        const next = [i + 1];\\n        for (let j = 0; j < expected.length; j++)\\n            next.push(Math.min(next[j] + 1, previous[j + 1] + 1, previous[j] + (actual[i] === expected[j] ? 0 : 1)));\\n        previous = next;\\n    }\\n    return previous[expected.length];\\n}\\nfunction commandLeadIn(text) {\\n    return text.trim().replace(/^(?:(?:вот|ну|пожалуйста|слушай|давай)[\\\\s,.:;!—-]+){1,3}/iu, \'\');\\n}\\n// Modes have explicit on/off semantics. Parse complete phrases before music\\n// extraction; a model trained only on player controls cannot supply these labels.\\nfunction modeCommand(text) {\\n    const words = normalizeSpeech(commandLeadIn(text)).replace(/ (?:пожалуйста|please)$/u, \'\');\\n    const on = \'(?:включи|включить|включай|ключи|ключить|запусти|запустить|поставь|поставить|активируй|enable|turn on)\';\\n    const off = \'(?:выключи|выключить|отключи|отключить|отключай|деактивируй|убери|останови|disable|turn off)\';\\n    const auto = \'(?:бесконечный режим(?: воспроизведения)?|режим (?:бесконечного воспроизведения|автоподбора|автоплея|рекомендаций)|бесконечное воспроизведение|бесконечную музыку|музыку бесконечно|\'\\n        + \'авто ?плей|автопли|автоплэй|автоподбор(?: песен| музыки)?|автовоспроизведение|автоматический подбор(?: песен| музыки)?|\'\\n        + \'автоматическое воспроизведение|автоматический режим|рекомендации(?: песен| музыки)?|autoplay|infinite mode)\'\\n        + \'(?: (?:от|с|на) (?:youtube|ютуба|ютубе))?\';\\n    const loop = \'(?:(?:повтор|повторение)(?: (?:трека|песни|этого трека|этой песни))?|режим (?:повтора|повторения)(?: трека| песни)?|\'\\n        + \'зацикливание(?: трека| песни)?|repeat|loop)\';\\n    const voice = \'(?:голосовое управление|голосовой режим|режим голосового управления|прослушивание)\';\\n    const match = (verb, target) => new RegExp(`^${verb} ${target}$`, \'u\').test(words);\\n    const unsafe = text.includes(\'?\') || /(?:^| )(?:не|нет|если|и|или|потом|затем|not|never|and|or|then)(?: |$)/u.test(words);\\n    if (!unsafe) {\\n        if (new RegExp(`^${on} громкость(?: |$)`, \'u\').test(words)) {\\n            const level = parseVolume(text);\\n            return level === null ? { action: \'unknown\' } : { action: \'volume_set\', level };\\n        }\\n        if (match(on, auto))\\n            return { action: \'autoplay_on\' };\\n        if (match(off, auto))\\n            return { action: \'autoplay_off\' };\\n        if (match(on, loop) || /^(?:повторяй|зацикли) (?:этот трек|эту песню|трек|песню)$/u.test(words))\\n            return { action: \'loop_on\' };\\n        if (match(off, loop) || /^перестань повторять (?:этот трек|эту песню|трек|песню)$/u.test(words))\\n            return { action: \'loop_off\' };\\n        if (match(on, voice))\\n            return { action: \'voice_on\' };\\n        if (match(off, voice))\\n            return { action: \'voice_off\' };\\n        if (match(on, \'(?:паузу|режим паузы)\'))\\n            return { action: \'pause\' };\\n        if (match(off, \'(?:паузу|режим паузы)\'))\\n            return { action: \'resume\' };\\n        if (/^включи (?:следующий трек|следующую песню)$/u.test(words))\\n            return { action: \'skip\' };\\n        if (/^(?:очисти|очистить|сбрось|сбросить) очередь(?: песен| треков)?$|^удали (?:все )?(?:песни|треки) из очереди$/u.test(words))\\n            return { action: \'queue_clear\' };\\n    }\\n    // Reserve mode requests even when negated, compound, incomplete or unknown.\\n    // Explicit titles (\\"включи песню Бесконечный режим\\") stay music queries.\\n    const target = words.replace(new RegExp(`^(?:(?:не|нет) )?(?:${on}|${off}) `, \'u\'), \'\');\\n    if (new RegExp(`^(?:(?:${auto}|${loop}|${voice})(?: |$)|режим(?: |$)|[\\\\\\\\p{L}]+ режим(?: |$)|паузу(?: |$))`, \'u\').test(target)\\n        || (target !== words && /^громкость(?: |$)/u.test(target))\\n        || /^(?:очисти|очистить|сбрось|сбросить|удали|удалить|повторяй|зацикли|перестань повторять)(?: |$)/u.test(words)) {\\n        return { action: \'unknown\' };\\n    }\\n    return null;\\n}\\nfunction controlAlias(text) {\\n    if (text.includes(\'?\'))\\n        return null;\\n    const words = normalizeSpeech(commandLeadIn(text)).replace(/ (?:пожалуйста|please)$/u, \'\');\\n    if (/^(?:следующ(?:ий|ая|ее|ую)|следущий|следующе|далее|дальше|переключи|переключись|скип|скипни|некст|next|skip)(?: (?:трек|песню|песня|track|song))?$|^пропусти$/u.test(words))\\n        return \'Следующий трек\';\\n    if (/^(?:pause|пауза|паузу)$/u.test(words))\\n        return \'Поставь на паузу\';\\n    if (/^(?:resume|continue)$/u.test(words))\\n        return \'Продолжи музыку\';\\n    if (/^(?:stop)$/u.test(words))\\n        return \'Стоп\';\\n    if (/^(?:louder)$/u.test(words))\\n        return \'Сделай громче\';\\n    if (/^(?:quieter)$/u.test(words))\\n        return \'Сделай тише\';\\n    return null;\\n}\\n// Closed control prefixes take priority over a free-form YouTube query.\\nfunction playerControlRequest(text) {\\n    if (modeCommand(text))\\n        return true;\\n    const words = normalizeSpeech(controlAlias(text) ?? commandLeadIn(text));\\n    return /^(?:следующ\\\\p{L}*|пропуст\\\\p{L}*|скип\\\\p{L}*|пауз\\\\p{L}*|продолж\\\\p{L}*|возобнов\\\\p{L}*|приостанов\\\\p{L}*|останов\\\\p{L}*|стоп|выключ\\\\p{L}*|отключ\\\\p{L}*|выйди|выйти|громк\\\\p{L}*|громче|тише|погромче|потише|установ\\\\p{L}*|увелич\\\\p{L}*|уменьш\\\\p{L}*|сдела\\\\p{L}*)(?: |$)/u.test(words)\\n        || /^(?:поставь|поставить) (?:(?:на )?паузу|громкость|звук)(?: |$)|^сними с паузы(?: |$)|^хватит играть(?: |$)|^на паузу(?: |$)|^музыка на паузе(?: |$)/u.test(words);\\n}\\n// Generic \\"turn it on\\" means resume only when an existing track is paused.\\n// Keep song titles and any extra words outside this closed set.\\nfunction contextualCommand(text, paused) {\\n    const cleaned = commandLeadIn(text);\\n    const alias = controlAlias(cleaned);\\n    if (alias)\\n        return alias;\\n    const words = normalizeSpeech(cleaned);\\n    if (paused && !text.includes(\'?\') && (/^(включи|включить|включай)( музыку| воспроизведение)?( снова| обратно| дальше)?$/.test(words)\\n        || /^(продолжай|продолжи|продолжить|возобнови|возобновить)( музыку| воспроизведение| играть)?$/.test(words))) {\\n        return \'Продолжи музыку\';\\n    }\\n    return cleaned;\\n}\\n// A closed decision model cannot supply arbitrary arguments. Parse numbers in code.\\nconst ONES = {\\n    ноль: 0, один: 1, одна: 1, два: 2, две: 2, три: 3, четыре: 4, пять: 5,\\n    шесть: 6, семь: 7, восемь: 8, девять: 9, десять: 10, одиннадцать: 11,\\n    двенадцать: 12, тринадцать: 13, четырнадцать: 14, пятнадцать: 15,\\n    шестнадцать: 16, семнадцать: 17, восемнадцать: 18, девятнадцать: 19,\\n    двадцать: 20, тридцать: 30, сорок: 40, пятьдесят: 50, шестьдесят: 60,\\n    семьдесят: 70, восемьдесят: 80, девяносто: 90, сто: 100,\\n};\\nfunction parseVolume(text) {\\n    // Reject decimals, signs and multiple separate values instead of clamping a guess.\\n    if (/\\\\d\\\\s*[.,]\\\\s*\\\\d|[-−+]\\\\s*\\\\d/u.test(text))\\n        return null;\\n    const tokens = normalizeSpeech(text).split(\' \');\\n    const groups = [];\\n    let group = [];\\n    for (const token of tokens) {\\n        const value = /^\\\\d+$/.test(token) ? Number(token) : ONES[token];\\n        if (value !== undefined)\\n            group.push(value);\\n        else if (group.length) {\\n            groups.push(group);\\n            group = [];\\n        }\\n    }\\n    if (group.length)\\n        groups.push(group);\\n    if (groups.length !== 1)\\n        return null;\\n    const parts = groups[0];\\n    const numberTokens = tokens.filter((token) => /^\\\\d+$/.test(token) || ONES[token] !== undefined);\\n    if (numberTokens.some((token) => /^\\\\d+$/.test(token)) && numberTokens.length !== 1)\\n        return null;\\n    if (parts.length > 3 || parts.some((n, i) => i > 0 && (n <= 0 || n >= parts[i - 1]\\n        || parts[i - 1] < 20 || (parts[i - 1] < 100 && (n >= 10 || parts[i - 1] % 10 !== 0)))))\\n        return null;\\n    const level = parts.reduce((sum, n) => sum + n, 0);\\n    return Number.isInteger(level) && level >= 1 && level <= 150 ? level : null;\\n}\\nfunction unsupportedSpeech(text) {\\n    const mode = modeCommand(text);\\n    if (mode)\\n        return mode.action === \'unknown\';\\n    const words = normalizeSpeech(text).split(\' \');\\n    if (!words[0] || words.length > 35\\n        || words.some((word) => /^(не|нет|если|потом|затем|сначала|и|или|но|not|no|never|dont|and|or|then|but)$/.test(word))\\n        || /^(?:don t|do not|please don t)(?: |$)/u.test(words.join(\' \')))\\n        return true;\\n    if (/\\\\?/u.test(text) || /^(как|кто|когда|почему|зачем|расскажи|покажи|привет|спасибо|мне нравится|музыка на паузе)(?: |$)/u.test(words.join(\' \')))\\n        return true;\\n    if (words.some((word) => /^(добавь|найди|поиск|сыграй|воспроизведи|очисти|плейлист|песн\\\\p{L}*)$/u.test(word)))\\n        return true;\\n    if (words.includes(\'включи\') || words.includes(\'включить\'))\\n        return true;\\n    if (words.includes(\'поставь\') || words.includes(\'поставить\')) {\\n        if (!/^(поставь|поставить) (на )?паузу$/.test(words.join(\' \')) && !words.includes(\'громкость\'))\\n            return true;\\n    }\\n    if (words.some((word) => /^трек\\\\p{L}*$/u.test(word)) && !words.some((word) => /^(следующий|пропусти|пропустить|скипни)$/.test(word)))\\n        return true;\\n    return false;\\n}\\nfunction validateIntent(text, decision, threshold = 0.6) {\\n    const direct = modeCommand(text);\\n    if (direct)\\n        return direct;\\n    if (unsupportedSpeech(text) || !Number.isFinite(decision.confidence)\\n        || decision.confidence < threshold || decision.confidence > 1)\\n        return { action: \'unknown\' };\\n    const allowed = [\'skip\', \'pause\', \'resume\', \'stop\', \'volume_set\', \'volume_up\', \'volume_down\', \'unknown\'];\\n    if (!allowed.includes(decision.action))\\n        return { action: \'unknown\' };\\n    // Probabilities are not calibrated: never execute an action contradicting an explicit verb.\\n    const words = normalizeSpeech(text).split(\' \');\\n    const incompatible = {\\n        skip: /^(пауз\\\\p{L}*|продолж\\\\p{L}*|возобнов\\\\p{L}*|останов\\\\p{L}*|отключ\\\\p{L}*|громк\\\\p{L}*|громче|тише)$/u,\\n        pause: /^(следующ\\\\p{L}*|пропуст\\\\p{L}*|пропусти|продолж\\\\p{L}*|возобнов\\\\p{L}*|сними|останов\\\\p{L}*|отключ\\\\p{L}*|громче|тише)$/u,\\n        resume: /^(следующ\\\\p{L}*|пропуст\\\\p{L}*|пропусти|приостанов\\\\p{L}*|останов\\\\p{L}*|отключ\\\\p{L}*|громче|тише)$/u,\\n        stop: /^(следующ\\\\p{L}*|пропуст\\\\p{L}*|пропусти|приостанов\\\\p{L}*|пауз\\\\p{L}*|продолж\\\\p{L}*|возобнов\\\\p{L}*|сними|громче|тише)$/u,\\n    };\\n    const contradiction = incompatible[decision.action];\\n    if (contradiction && words.some((word) => contradiction.test(word)))\\n        return { action: \'unknown\' };\\n    if (decision.action === \'resume\' && words.some((word) => /^постав\\\\p{L}*$/u.test(word)))\\n        return { action: \'unknown\' };\\n    if (decision.action === \'stop\' && !words.some((word) => /^(музык\\\\p{L}*|воспроизвед\\\\p{L}*|бот\\\\p{L}*|канал\\\\p{L}*|плеер\\\\p{L}*|звук)$/u.test(word))\\n        && !/^(стоп|останови|остановить|отключись|выключись|выйди|хватит играть)$/.test(words.join(\' \')))\\n        return { action: \'unknown\' };\\n    const hasNumber = words.some((word) => /^\\\\d+$/.test(word) || ONES[word] !== undefined);\\n    if (decision.action.startsWith(\'volume_\')) {\\n        if (words.some((word) => /^(следующ\\\\p{L}*|пропуст\\\\p{L}*|пропусти|приостанов\\\\p{L}*|пауз\\\\p{L}*|продолж\\\\p{L}*|возобнов\\\\p{L}*|останов\\\\p{L}*|отключ\\\\p{L}*)$/u.test(word)))\\n            return { action: \'unknown\' };\\n        if (hasNumber && !words.includes(\'до\') && words.some((word) => /^(громче|тише|увелич\\\\p{L}*|уменьш\\\\p{L}*)$/u.test(word)))\\n            return { action: \'unknown\' };\\n    }\\n    if (decision.action === \'volume_set\') {\\n        const level = parseVolume(text);\\n        return level === null ? { action: \'unknown\' } : { action: \'volume_set\', level };\\n    }\\n    if ((decision.action === \'volume_up\' || decision.action === \'volume_down\') && hasNumber)\\n        return { action: \'unknown\' };\\n    return { action: decision.action };\\n}\\n//# sourceMappingURL=intents.js.map\\nfunction musicVariant(query) {\\n    const words = normalizeSpeech(query);\\n    for (const [variant, pattern] of [\\n        [\'live\', /(?:^| )(?:live|концертн\\\\p{L}*)(?: |$)/u], [\'remix\', /(?:^| )(?:remix|ремикс\\\\p{L}*)(?: |$)/u],\\n        [\'cover\', /(?:^| )(?:cover|кавер\\\\p{L}*)(?: |$)/u], [\'acoustic\', /(?:^| )(?:acoustic|акустическ\\\\p{L}*)(?: |$)/u],\\n        [\'instrumental\', /(?:^| )(?:instrumental|инструментальн\\\\p{L}*)(?: |$)/u],\\n    ])\\n        if (pattern.test(words))\\n            return variant;\\n    return null;\\n}\\nfunction musicState(message, request, player) {\\n    return { message, selected_track: null, parsed_request: { kind: \'play\',\\n            search_query: request.kind === \'search\' ? request.query : null,\\n            ...(request.kind === \'video\' ? { url: request.query } : {}), request_variant: musicVariant(request.query) },\\n        ...(player ? { player } : {}) };\\n}\\n// Extract from the original transcript: speech normalization would destroy URLs\\n// and punctuation in song titles. Only command boundaries are interpreted here.\\nfunction extractMusicRequest(message, allowBare = false) {\\n    if (message.length > 1000)\\n        return null;\\n    // Conversational lead-ins and punctuation inserted by ASR are not part of\\n    // the title. Stay anchored: never fish a command out of a longer sentence.\\n    const cleaned = commandLeadIn(message);\\n    if (playerControlRequest(cleaned))\\n        return null;\\n    // ASR can lose the quiet initial consonant in \\"включи\\". Interpret this\\n    // leading command form only; never rewrite words inside the query.\\n    const match = /^(?:(?:можешь|можете)\\\\s+)?(?:включи|включить|включай|ключи|ключить|поставь|поставить|сыграй|сыграть|воспроизведи|воспроизвести|запусти|запустить|проиграй|проиграть|хочу послушать|послушаем)(?:\\\\s*[,:\\\\u2014-]\\\\s*|\\\\s+)(.+)$/iu.exec(cleaned);\\n    if (!match && !allowBare)\\n        return null;\\n    if (!match) {\\n        const bare = normalizeSpeech(cleaned);\\n        // Only the speaker\'s command window permits a bare title/artist. Do not\\n        // turn conversations, negations, questions or incomplete commands into music.\\n        if (!bare || bare.split(\' \').length > 35 || cleaned.includes(\'?\')\\n            || /^(?:не|нет|если|сначала|потом|затем|как|кто|когда|почему|зачем|расскажи|покажи|привет|спасибо|я|мы|ты|вы|он|она|мне нравится|добавь|найди|очисти|not|no|never|don t|dont|do not|please don t)(?: |$)/u.test(bare)\\n            || /(?:^| )(?:next|skip|pause|resume|stop|louder|quieter)(?: |$)/u.test(bare)\\n            || /(?:^| )(?:включи|включить|включай|ключи|ключить|поставь|поставить|сыграй|сыграть|воспроизведи|воспроизвести|запусти|запустить|проиграй|проиграть)(?: |$)/u.test(bare))\\n            return null;\\n    }\\n    const target = (match?.[1] ?? cleaned).trim();\\n    if (/^(?:песню|трек)\\\\s+из[.!…]*$/iu.test(target))\\n        return null;\\n    // \\"Song from Luntik\\" needs its music noun: searching only \\"from Luntik\\"\\n    // predominantly returns cartoon episodes, unlike an explicit song title.\\n    const query = /^(?:песню|трек)\\\\s+из\\\\s+/iu.test(target)\\n        ? target.replace(/^песню\\\\s+/iu, \'песня \')\\n        : target.replace(/^(?:песню|трек)\\\\s+/iu, \'\').trim();\\n    const words = normalizeSpeech(query);\\n    if (!words || /^(?:на паузу|паузу|музыку(?: снова| обратно| дальше)?|музыка|воспроизведение|снова|обратно|дальше|песню|песни|песня|трек|эту(?: песню)?|этот(?: трек)?|ее|его|это|то|ту|тот|первую|первый|следующую(?: песню| музыку)?|следующий(?: трек)?)$/u.test(words))\\n        return null;\\n    // Conjunctions in names (e.g. \\"Numb и Encore\\") are valid. A second command\\n    // verb after a command separator is not a song name.\\n    if (/(?:^|\\\\s)(?:и|или|потом|затем)\\\\s+(?:не\\\\s+)?(?:включ\\\\p{L}*|ключи\\\\p{L}*|постав\\\\p{L}*|сыгра\\\\p{L}*|воспроизвед\\\\p{L}*|запуст\\\\p{L}*|проигра\\\\p{L}*|останов\\\\p{L}*|останавли\\\\p{L}*|выключ\\\\p{L}*|пропуст\\\\p{L}*|продолж\\\\p{L}*|возобнов\\\\p{L}*|сдела\\\\p{L}*|пауза|следующий|громче|тише|стоп)(?:\\\\s|$)/iu.test(words))\\n        return null;\\n    if (/[,;.!]\\\\s*(?:не\\\\s+)?(?:включ\\\\p{L}*|ключи\\\\p{L}*|постав\\\\p{L}*|сыгра\\\\p{L}*|воспроизвед\\\\p{L}*|запуст\\\\p{L}*|проигра\\\\p{L}*|останов\\\\p{L}*|выключ\\\\p{L}*|пропуст\\\\p{L}*|продолж\\\\p{L}*|возобнов\\\\p{L}*|сдела\\\\p{L}*|пауза|следующий|громче|тише|стоп)(?:\\\\s|$)/iu.test(query))\\n        return null;\\n    if (/https?:\\\\/\\\\//iu.test(query)) {\\n        let url;\\n        try {\\n            url = new URL(query.replace(/[.!…]+$/u, \'\'));\\n        }\\n        catch {\\n            return null;\\n        }\\n        if (![\'http:\', \'https:\'].includes(url.protocol)\\n            || ![\'youtube.com\', \'www.youtube.com\', \'music.youtube.com\', \'m.youtube.com\', \'youtu.be\'].includes(url.hostname.toLowerCase()))\\n            return null;\\n        const id = url.hostname.toLowerCase() === \'youtu.be\' ? url.pathname.slice(1).split(\'/\')[0] : url.searchParams.get(\'v\');\\n        if (!id || !/^[\\\\w-]{11}$/.test(id))\\n            return null;\\n        return { kind: \'video\', query: `https://www.youtube.com/watch?v=${id}` };\\n    }\\n    return { kind: \'search\', query };\\n}\\nfunction validConfidence(value) {\\n    return Number.isFinite(value) && value >= 0.6 && value <= 1;\\n}\\n\\nimport { readFileSync } from \'node:fs\';\\nconst rows = JSON.parse(readFileSync(0, \'utf8\'));\\nconst output = rows.map(row => {\\n  const canonical = contextualCommand(row.message, Boolean(row.player?.paused));\\n  const direct = modeCommand(canonical);\\n  const explicit = extractMusicRequest(canonical);\\n  const music = explicit ?? extractMusicRequest(canonical, true);\\n  const unsupported = !music && unsupportedSpeech(canonical);\\n  const model_required = !unsupported && !explicit && !direct;\\n  let final_action = direct?.action ?? \'unknown\';\\n  if (unsupported) final_action = \'unknown\';\\n  else if (!direct && !explicit && row.decision) {\\n    final_action = playerControlRequest(canonical) ? validateIntent(canonical, row.decision).action : \'unknown\';\\n  }\\n  if (!unsupported && final_action === \'unknown\' && music) final_action = \'play\';\\n  return {canonical, direct_action: direct?.action ?? null, music_kind: music?.kind ?? null,\\n          unsupported, model_required, final_action,\\n          wake_match: typeof row.wake_name === \'string\' ? isWakePhrase(row.message, row.wake_name) : null};\\n});\\nprocess.stdout.write(JSON.stringify(output));\\n", "source_sha256": {"src/voice/intents.ts": "0e12fc29ada4b2ab5df4216513398246ab873bf58e6c920f249c89009f8d24ea", "src/voice/music.ts": "abe27c9aba28062c5ad71d8c45a8a21099c95cdc6ff37ad56a653907667debdc", "src/voice/session.ts": "b927957d147e1e230fc9b27a8ec09ba399eaa254fe821fe6253b3f55b70ca274", "src/voice/GuildVoice.ts": "22339b6e3fd03b679dfbe16e15c5e3da6021491c87b0b4ff53a7afead70ff375", "src/voice/training.ts": "f534614f359a3c537c7d936ee722d7cbd33e9df250e65990e6bd4799f6444f00", "src/voice/export-training.ts": "3c51f5d2f23fcc145a69910c188f16847b28728ac1ec74e46df9b8aadf4bbc15", "scripts/voice_worker.py": "7392d40ef3e5f024f623e8aaf60b3c351e0839dd137fbda3f65a5c094e09d7c2", "scripts/laya_music_v3.json": "7828f6e96bcf8445f18c2ed6b3f3835e72fdb10015c3b2ae26d475ad303260be", "scripts/laya_training.py": "1c28a298ef754a957b3b534abeaaa273f4e0e0317c68560120d0eeef52ab4b00"}}')

scripts_dir = WORKDIR / "contract" / "scripts"
scripts_dir.mkdir(parents=True, exist_ok=True)
helper_path = scripts_dir / "laya_training.py"
worker_path = scripts_dir / "voice_worker.py"
bridge_path = scripts_dir / "bot_parser.mjs"
helper_path.write_text(SNAPSHOT["helper"], encoding="utf-8")
worker_path.write_text(SNAPSHOT["worker"], encoding="utf-8")
bridge_path.write_text(SNAPSHOT["bridge"], encoding="utf-8")
(scripts_dir / "laya_music_v3.json").write_text(json.dumps(SNAPSHOT["music_schema"], ensure_ascii=False), encoding="utf-8")
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
training = load_module("lia_training", helper_path)
worker = load_module("lia_worker_contract", worker_path)  # Не загружает Whisper и TTS.
INTENT_QUESTIONS = SNAPSHOT["intent_questions"]
PLANNER_QUESTIONS = training.planner_questions(INTENT_QUESTIONS)
print("Runtime action:", list(worker.QUESTIONS["action"]["criteria"]))
print("Reviewed intent:", list(INTENT_QUESTIONS["intent"]["criteria"]))


## 6. Загрузка и разделение без утечки

Четыре независимые части: train 60%, validation 15% для выбора эпохи,
calibration 10% для температуры и test 15% для окончательной оценки.
Доли приблизительные: группы неделимы. Варианты одной фразы и оба prompt остаются вместе.
Корпуса runtime и planner группируются совместно, включая одинаковые исходные и приведённые тексты.
Если классов/семейств недостаточно, выполнение остановится с объяснением.
Test не участвует в выборе эпохи, температуры или порогов.


In [ ]:
if SMOKE_TEST:
    # Искусственные данные только для технической проверки, никаких заявлений о качестве.
    action_texts = {"skip": "пропусти", "pause": "поставь на паузу", "resume": "продолжи музыку",
                    "stop": "останови музыку", "volume_set": "громкость 70",
                    "volume_up": "сделай громче", "volume_down": "сделай тише",
                    "unknown": "не выключай музыку", "play": "включи песню Numb"}
    smoke_rows = []
    for label, phrase in action_texts.items():
        for family in range(12):
            smoke_rows.append({"event_id": f"smoke-{label}-{family}", "message": f"{phrase} номер {family}",
                               "intent": label, "reviewed": True, "group_id": f"{label}-{family}"})
    smoke_path = WORKDIR / "commands-reviewed.jsonl"
    smoke_path.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in smoke_rows) + "\n", encoding="utf-8")
    DATA_PATHS = [str(smoke_path)]
    EPOCHS, BATCH_SIZE, ACCUMULATION = 1, 2, 2
if not DATA_PATHS:
    found = set()
    for root in (Path("/content/data"), Path("/content"), Path("/kaggle/input")):
        if root.exists():
            names = ["reviewed-nli.jsonl", "commands-reviewed.jsonl"]
            if ALLOW_AUTOMATIC_LABELS:
                names += ["commands-synthetic.jsonl", "bot-commands-labeled.jsonl"]
            for name in names:
                found.update(root.rglob(name))
    DATA_PATHS = sorted(str(p) for p in found if WORKDIR not in p.parents)
if not DATA_PATHS:
    raise FileNotFoundError("Upload reviewed-nli.jsonl/commands-reviewed.jsonl, or fill DATA_PATHS")
commands = training.load_commands(DATA_PATHS, REVIEWS_PATH, INTENT_QUESTIONS["intent"]["criteria"], ALLOW_AUTOMATIC_LABELS)
commands = training.prepare_commands(commands, bridge_path)
runtime_labels = set(worker.QUESTIONS["action"]["criteria"])
present_labels = {r["intent"] if r["intent"] in runtime_labels else "unknown" for r in commands}
if runtime_labels - present_labels:
    raise ValueError(f"Missing runtime action classes: {sorted(runtime_labels - present_labels)}; add reviewed examples")
if AUTO_LOAD_PLANNER and not PLANNER_PATHS:
    PLANNER_PATHS = sorted({str(Path(path).parent / "planner-synthetic.jsonl") for path in DATA_PATHS
                            if (Path(path).parent / "planner-synthetic.jsonl").is_file()})
    if not PLANNER_PATHS:
        raise FileNotFoundError("AUTO_LOAD_PLANNER=True but no planner-synthetic.jsonl beside command files")
planner_rows = training.load_planner(PLANNER_PATHS, PLANNER_QUESTIONS, bridge_path, ALLOW_AUTOMATIC_LABELS) if PLANNER_PATHS else []
if planner_rows:
    processed = training.node_bridge([{"message": r["message"], "player": r["state"]["player"]} for r in planner_rows], bridge_path)
    for row, parsed in zip(planner_rows, processed, strict=True):
        row["canonical"] = parsed["canonical"]
all_rows = commands + planner_rows
if len({r["event_id"] for r in all_rows}) != len(all_rows):
    raise ValueError("Duplicate event_id across runtime/planner corpora")
# Joint grouping prevents leakage when a phrase appears in both tasks or several contexts.
all_splits = training.grouped_split(all_rows, SEED, fractions=(0.4, 0.2, 0.2, 0.2) if SMOKE_TEST else (0.6, 0.15, 0.1, 0.15))
command_splits = {name: [r for r in rows if r.get("task") != "planner"] for name, rows in all_splits.items()}
planner_splits = {name: [r for r in rows if r.get("task") == "planner"] for name, rows in all_splits.items()} if planner_rows else None
record_splits = {name: training.runtime_records(rows, worker) for name, rows in command_splits.items()}
# Сохраняем ID/группы, не тексты участников, в отчёт экспорта.
split_manifest = {name: [{"event_id": r["event_id"], "group_id": r["group_id"], "intent": r["intent"]} for r in rows]
                  for name, rows in command_splits.items()}
for name, rows in command_splits.items():
    print(name, len(rows), dict(training.Counter(r["intent"] for r in rows)),
          "model_required:", sum(r["model_required"] for r in rows))
print("Data origins:", dict(training.Counter(r["origin"] for r in all_rows)))
if ALLOW_AUTOMATIC_LABELS:
    print("Automatic labels enabled: synthetic/rule-labelled data is not human-reviewed ground truth")


## 7. Дополнительно: реагировать и выбирать функцию

`PLANNER_PATHS=[]` по умолчанию. Эти вопросы **не вызываются текущим ботом**.
Они позволяют обучать тот же checkpoint для будущего расширения без изменения runtime action.
Нужен отдельный вручную размеченный корпус, включая фон, неверное имя, чужого говорящего,
выключенный голос, неподдерживаемую просьбу после активации и нормальные команды.
Экспорт журнала команд недостаточен: в нём нет idle/owner/voice_enabled.

Пример записи (не выдаётся за обучающий корпус):
```json
{"event_id":"planner-1","group_id":"pause-context-family","reviewed":true,"state":{"message":"поставь на паузу","phase":"awaiting","voice_enabled":true,"in_bot_channel":true,"is_owner":true,"wake_name":"Муза","player":{"connected":true,"playing":true,"paused":false,"autoplay":false,"queue_length":1}},"gold":{"should_respond":"respond","next_tool":"player_control","intent":"pause"}}
```
`should_respond`: respond/ignore. `next_tool`: wake_ack/youtube_music_search/direct_youtube_video/player_control/no_tool.
`intent`: те же 16 меток ручной разметки. В idle допустимо отдельное имя → respond/wake_ack/unknown;
фон, чужой канал, disabled и busy → ignore/no_tool/unknown; неподдерживаемая реплика владельца
в awaiting → respond/no_tool/unknown (текущее голосовое подтверждение unknown).
В awaiting запрос песни → respond/search или direct_video/play; режим → respond/player_control/его intent.
Плейлист как отдельный инструмент не поддерживается голосовым путём; selected/current_track не используются.

`group_id` объединяет контекстные варианты одной реплики. Для каждой комбинации gold нужен набор
независимых семей; иначе проверка покрытия остановит обучение. Корпус planner разделяется до обучения.
Даже после подключения модели проверки канала, доступа, владельца, таймеров и отмены остаются в коде.


In [ ]:
if PLANNER_PATHS:
    for name, rows in planner_splits.items():
        record_splits[name].extend(rows)
    split_manifest["planner"] = {name: [{"event_id": r["event_id"], "group_id": r["group_id"]} for r in rows]
                                 for name, rows in planner_splits.items()}
print("Planner training:", bool(PLANNER_PATHS))


## 8. Модель и токенизация

Загружается существующая архитектура Laya со scorer по маркерам вариантов, не
`AutoModelForSequenceClassification` с несовместимыми весами. Act/escalate-head отдельно не обучается:
это не поле `action` музыкального бота. Сокращённые и основные вопросы берутся из обработчика.
Температуры базы обнуляются до 1 для честного сравнения некалиброванных логитов.


In [ ]:
agent = laya.load(BASE_MODEL, device=DEVICE, revision=BASE_REVISION)
agent.temperature = [1.0, 1.0, 1.0]
agent.temperature_by_options = {}
agent.lang_temperatures = {}
# T4 не поддерживает BF16; RTX/A100 можно использовать BF16.
agent.dtype = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
agent.cfg["amp_dtype"] = "bf16" if agent.dtype == torch.bfloat16 else "fp16"
agent.cfg["temperature"] = [1.0, 1.0, 1.0]
agent.cfg["temperature_by_options"] = {}
training.batches.pad_id = agent.tok.pad_token_id
encoded = {name: training.encode_records(agent, records) for name, records in record_splits.items()}
print({name: {"questions": len(items), "max_tokens": max(len(x["ids"]) for x in items)} for name, items in encoded.items()})


## 9. Baseline и обучение

Baseline считается на validation. Лучший checkpoint выбирается по validation NLL;
test остаётся нетронутым до финальной оценки. Encoder по умолчанию заморожен, его dropout выключен.
Если реальных данных достаточно и validation перестал улучшаться, отдельным экспериментом
включите TRAIN_ENCODER и сравните на validation. Не выбирайте конфигурацию по test.


In [ ]:
baseline_validation, _ = training.evaluate_logits(training.raw_logits(agent, encoded["validation"], BATCH_SIZE))
print("Baseline validation:", json.dumps(baseline_validation, ensure_ascii=False, indent=2))
history = training.train(agent, encoded["train"], encoded["validation"], WORKDIR / "best",
                         epochs=EPOCHS, batch_size=BATCH_SIZE, accumulation=ACCUMULATION,
                         head_lr=HEAD_LR, encoder_lr=ENCODER_LR, train_encoder=TRAIN_ENCODER, seed=SEED)


## 10. Калибровка уверенности

Используется только calibration. Для каждого события fallback-вопрос не удваивает объём.
Laya 0.3.22 требует минимум 10 независимых записей для общей температуры типа и 2000 для
отдельного bucket. При меньшем числе сохраняется 1; это не доказательство калиброванности.
Confidence в боте — `answer_confidence=max(probabilities)`, не entropy-confidence и не act_probability.
Фиксированные runtime пороги сохраняются: 0.6, для слабого fallback 0.75.
Порог нельзя подбирать на test; исследовать его следует в отдельном эксперименте на validation.


In [ ]:
calibration_rows = training.raw_logits(agent, encoded["calibration"], BATCH_SIZE)
temperatures = training.calibrate(calibration_rows)
print("Independent calibration events:", len({x["item"]["event_id"] for x in calibration_rows}))
print("Temperatures:", temperatures)
agent.temperature = temperatures["temperature"]
agent.temperature_by_options = temperatures["temperature_by_options"]
test_logits = training.raw_logits(agent, encoded["test"], BATCH_SIZE)
uncalibrated_test, _ = training.evaluate_logits(test_logits)
calibrated_test, test_predictions = training.evaluate_logits(test_logits, temperatures)
pipeline_metrics, pipeline_rows = training.runtime_evaluation(agent, command_splits["test"], worker, bridge_path)
print("Test:", json.dumps(calibrated_test, ensure_ascii=False, indent=2))
print("Bot text pipeline:", pipeline_metrics)


## 11. Проверка качества

Macro-F1, precision/recall по классам, NLL, Brier и ECE относятся к выбору вариантов.
Отдельно смотрите `model_required_accuracy`: общий pipeline включает команды, выполненные правилами,
и может скрывать слабую модель. False-control-rate — доля play/unknown, ставших управляющей командой.
Не путайте отказ классификатора с необходимостью молчать: после активации unknown может озвучиваться.
Маленький test не подтверждает надёжность; просмотрите ошибки и проверьте реальные записи отдельно.


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots()
ax.plot([x["epoch"] for x in history], [x["train_nll"] for x in history], "o-", label="Train NLL", color="#245b91")
ax.plot([x["epoch"] for x in history], [x["validation_nll"] for x in history], "s--", label="Validation NLL", color="#b47b24")
ax.set(xlabel="Эпоха", ylabel="NLL (меньше лучше)", title="Обучение: train и независимая validation")
ax.set_xticks([x["epoch"] for x in history])
ax.legend(); fig.tight_layout(); plt.show()
primary = [r for r in test_predictions if r["task"] == "action_primary"]
labels = list(worker.QUESTIONS["action"]["criteria"])
matrix = np.zeros((len(labels), len(labels)), dtype=int)
for row in primary:
    matrix[labels.index(row["gold"]), labels.index(row["prediction"])] += 1
fig, ax = plt.subplots(figsize=(10, 8))
plot = ax.imshow(matrix, cmap="Blues", vmin=0)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center", color="white" if matrix[i, j] > matrix.max() / 2 else "black")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
ax.set(xlabel="Предсказание", ylabel="Ручная метка", title=f"Primary action на test: {len(primary)} команд")
fig.colorbar(plot, ax=ax, label="Количество команд"); fig.tight_layout(); plt.show()
fig, ax = plt.subplots()
means, accuracies, sizes = [], [], []
for lo in np.arange(0, 1, 0.1):
    selected = [r for r in primary if lo <= r["confidence"] < lo + 0.1 or lo >= 0.9 and r["confidence"] == 1]
    if selected:
        means.append(np.mean([r["confidence"] for r in selected])); accuracies.append(np.mean([r["correct"] for r in selected])); sizes.append(len(selected))
ax.plot([0, 1], [0, 1], "--", color="#555555", label="Идеальная калибровка")
ax.plot(means, accuracies, "o-", color="#245b91", label="Наблюдаемая точность")
for x, y, n in zip(means, accuracies, sizes): ax.annotate(f"n={n}", (x, y), xytext=(4, 7), textcoords="offset points")
ax.set(xlim=(0, 1.02), ylim=(0, 1.08), xlabel="Средняя уверенность", ylabel="Доля верных ответов",
       title="Калибровка primary action на независимом test (bin=0.1)")
ax.legend(); fig.tight_layout(); plt.show()
print("Ошибки pipeline (ID без текста):", [r for r in pipeline_rows if r["gold"] != r["prediction"]][:30])


## 12. Экспорт и повторная загрузка

Экспорт содержит только веса, конфигурации, tokenizer и отчёт с ID/группами/метриками.
Исходные реплики участников не включаются. Проверяем совпадение ответов после загрузки с диска.
Архив не публикуется на Hugging Face автоматически.


In [ ]:
# Ячейку можно повторить после частичного экспорта: agent мог остаться на CPU.
if "reloaded" in globals():
    del reloaded
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
agent.device = torch.device(DEVICE)
agent.model.to(device=agent.device, dtype=agent.dtype).eval()
model_name = "laya-muz-bot-controls-v1" + ("-planner" if PLANNER_PATHS else "")
export_dir = WORKDIR / (model_name + ("-SMOKE-ONLY" if SMOKE_TEST else ""))
training.save_checkpoint(agent, export_dir, model_name, temperatures)
probe_texts = [r["canonical"] for r in command_splits["test"][:5]]
before = [worker.classify(agent, text) for text in probe_texts]
# Освобождаем видеопамять до второй загрузки (важно для T4/8 GB).
agent.model.cpu()
agent.device = torch.device("cpu")
reloaded = laya.load(str(export_dir), device=DEVICE)
after = [worker.classify(reloaded, text) for text in probe_texts]
# GPU/FP16 может немного менять округлённую уверенность после загрузки.
assert all(a["action"] == b["action"] and abs(a["confidence"] - b["confidence"]) <= 0.005
           for a, b in zip(before, after, strict=True)), "Checkpoint round-trip changed runtime decisions"
if PLANNER_PATHS:
    example = planner_splits["test"][0]
    planner_answers = reloaded.predict(example["state"], PLANNER_QUESTIONS)["answers"]
    print("Experimental planner probe:", {k: v["choice"] for k, v in planner_answers.items()})
input_hashes = {Path(p).name + f"#{i}": training.sha256_file(p) for i, p in enumerate(DATA_PATHS + PLANNER_PATHS)}
if REVIEWS_PATH: input_hashes["reviews"] = training.sha256_file(REVIEWS_PATH)
report = {"smoke_only": SMOKE_TEST, "runtime_contract": "action + primary/fallback + TS guards",
          "planner_enabled_in_bot": False, "planner_trained": bool(PLANNER_PATHS),
          "automatic_labels_allowed": ALLOW_AUTOMATIC_LABELS,
          "evaluation_warning": "Synthetic/rule-labelled holdouts are not independently annotated real-user accuracy" if ALLOW_AUTOMATIC_LABELS else None,
          "data_origins": dict(training.Counter(r["origin"] for r in all_rows)),
          "source_sha256": SNAPSHOT["source_sha256"], "input_sha256": input_hashes,
          "split_manifest": split_manifest, "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
          "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "accumulation": ACCUMULATION,
          "head_lr": HEAD_LR, "encoder_lr": ENCODER_LR, "train_encoder": TRAIN_ENCODER,
          "history": history, "baseline_validation": baseline_validation,
          "uncalibrated_test": uncalibrated_test, "calibrated_test": calibrated_test,
          "pipeline_metrics": pipeline_metrics, "temperatures": temperatures,
          "environment": {k: importlib.metadata.version(k) for k in ("torch", "laya", "transformers", "huggingface-hub", "numpy")},
          "checkpoint_roundtrip": "passed"}
(export_dir / "training_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
checkpoint_hashes = {str(p.relative_to(export_dir)): training.sha256_file(p) for p in export_dir.rglob("*") if p.is_file()}
(export_dir / "sha256.json").write_text(json.dumps(checkpoint_hashes, indent=2), encoding="utf-8")
archive_path = shutil.make_archive(str(export_dir), "zip", root_dir=export_dir.parent, base_dir=export_dir.name)
print("Archive:", archive_path)
print("SMOKE ONLY — не подключать к боту" if SMOKE_TEST else "Проверьте метрики и ошибки перед подключением к боту")


## 13. Подключение

В Colab скачайте архив через `from google.colab import files; files.download(archive_path)`.
В Kaggle он находится в Output (`/kaggle/working/lia-training/`). Распакуйте обычный,
проверенный checkpoint в `.runtime/voice/models/laya-muz-bot-controls-v1/`.
В `.env` задайте:
```dotenv
LAYA_MODEL_PATH=.runtime/voice/models/laya-muz-bot-controls-v1
```
Перезапустите бота; проверьте имя модели в логе. Если voice runtime ещё не подготовлен,
выполните `npm run setup:voice`. Для смены весов `npm run deploy` не требуется.
Не переименовывайте `model_name` в `laya-muz-bot-ds-v3`: это активирует другой старый адаптер.
Для отката уберите `LAYA_MODEL_PATH` и перезапустите бот с базовой моделью из подготовленного кэша.

Перед использованием проверьте реальные команды, ложные обращения, уход говорящего, таймаут,
паузу/продолжение, числа 1/150/0/151, отрицания, новые названия песен и неизвестные просьбы.
Для новой planner-схемы нужна отдельная интеграция в VoiceSession/worker: сейчас она не вызывается.
Модель не заменяет проверки канала, владельца и AbortSignal; уверенность не даёт права выполнить команду.

Источники метода и API: [Laya upstream](https://github.com/NandhaKishorM/laya),
[официальный fine-tuning notebook](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb).
Автор этого ноутбука использует supervised CE и отдельную калибровку; это не копия upstream RLCD.
Схема и состояния проверены по локальному `laya==0.3.22` и исходникам muz-bot-ds.
